In [ ]:
%reload_ext autoreload
%autoreload 2

# Fourier integral operator in action
## 1D

In [ ]:
# ============================================================
# Frozen-Time FIO — 1D Example  (corrected)
# Schrödinger free propagator  exp(it·ξ²)
#
# FIX: for x-independent symbols, exp(itP) has the exact symbol
#      exp(it·ξ²). We build it directly — no Taylor truncation.
#
# WKB reference:  u(x,t) = exp(it·k₀²) · u₀(x)
# ============================================================

import sys
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, '.')        # adjust to your local path
from fio_bridge import PsiOpFIOBridge
from psiop     import PseudoDifferentialOperator

# ── Parameters ────────────────────────────────────────────────────────────────
LAM      = 40.0
K0       = 2.0
N_X      = 40
T_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25]
VERBOSE  = False

# ── Symbolic setup ────────────────────────────────────────────────────────────
x_sym  = sp.Symbol('x',  real=True)
xi_sym = sp.Symbol('xi', real=True)
y_sym  = sp.Symbol('y',  real=True)   # bridge integration variable

# WKB initial state  u₀(y) = exp(-y²/2) · exp(iλ k₀ y)
u0_phase_sym = K0 * y_sym
u0_amp_sym   = sp.exp(-y_sym**2 / 2)

x_grid = np.linspace(-2.5, 2.5, N_X)
u0_arr = np.exp(-x_grid**2 / 2) * np.exp(1j * LAM * K0 * x_grid)

bridge_kw = dict(lam=LAM, n_guesses=50, xi_range=(-10., 10.), verbose=VERBOSE)

# ── Frozen-time loop ──────────────────────────────────────────────────────────
snapshots = {}

for t_val in T_VALUES:
    # Exact symbol of exp(it·ξ²) — valid for all t, no Taylor truncation
    exp_sym = sp.exp(sp.I * t_val * xi_sym**2)
    exp_op  = PseudoDifferentialOperator(exp_sym, vars_x=[x_sym], mode='symbol')
    bridge  = PsiOpFIOBridge(exp_op, **bridge_kw)

    u_bridge = bridge.evaluate_grid(x_grid, u0_phase_sym, u0_amp_sym)

    # WKB reference: uniform phase shift exp(it·k₀²)
    u_ref = np.exp(1j * t_val * K0**2) * u0_arr

    err = float(np.max(np.abs(u_bridge - u_ref)) / (np.max(np.abs(u_ref)) + 1e-12))
    snapshots[t_val] = dict(bridge=u_bridge, ref=u_ref, err=err,
                            passed=(err < 5.0 / LAM))

# ── Visualization ─────────────────────────────────────────────────────────────
DARK_BG  = '#0d1117'
GRID_CLR = '#21262d'
REF_CLR  = '#58a6ff'
BRG_CLR  = '#f78166'
TICK_CLR = '#8b949e'
LBL_CLR  = '#c9d1d9'


n_snap = len(T_VALUES)
cols   = 3
rows   = (n_snap + cols - 1) // cols

fig = plt.figure(figsize=(16, 10), facecolor=DARK_BG)
fig.suptitle(
    r'Frozen-Time FIO · 1D Schrödinger  $e^{it\xi^2}$'
    f'   |   ' + r'$\lambda={:.0f}$,  $k_0={:.1f}$'.format(LAM, K0) + '\n'
    r'WKB ref: $u(x,t)=e^{itk_0^2}\,u_0(x)$   '
    r'— bridge symbol: $e^{it\xi^2}$ (exact, no Taylor)',
    color=LBL_CLR, fontsize=12, fontfamily='monospace', y=0.98,
)

gs = gridspec.GridSpec(rows, cols, figure=fig,
                       hspace=0.55, wspace=0.35,
                       left=0.06, right=0.97, top=0.88, bottom=0.10)

for idx, t_val in enumerate(T_VALUES):
    r  = snapshots[t_val]
    ax = fig.add_subplot(gs[idx // cols, idx % cols])
    ax.set_facecolor(DARK_BG)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_CLR)
    ax.tick_params(colors=TICK_CLR, labelsize=8)

    ax.plot(x_grid, np.real(r['ref']),    color=REF_CLR, lw=2.2,
            label='WKB ref  Re', zorder=3)
    ax.plot(x_grid, np.real(r['bridge']), color=BRG_CLR, lw=1.6,
            ls='--', label='bridge  Re', zorder=4)
    ax.plot(x_grid, np.imag(r['ref']),    color=REF_CLR, lw=1.0,
            alpha=0.35, label='WKB ref  Im')
    ax.plot(x_grid, np.imag(r['bridge']), color=BRG_CLR, lw=0.9,
            ls='--', alpha=0.35, label='bridge  Im')

    status = '✓ PASS' if r['passed'] else '✗ FAIL'
    status_color = '#3fb950' if r['passed'] else '#f85149'
    ax.set_title(f't = {t_val:.2f}     {status}     err = {r["err"]:.4f}',
                 color=status_color, fontsize=9, fontfamily='monospace')
    ax.set_xlabel('x', color=LBL_CLR, fontsize=8)
    ax.grid(True, color=GRID_CLR, lw=0.6, alpha=0.8)
    if idx == 0:
        ax.legend(fontsize=7, facecolor='#161b22', labelcolor=LBL_CLR,
                  edgecolor=GRID_CLR, loc='upper right')

# ── Error strip ───────────────────────────────────────────────────────────────
ax_err = fig.add_axes([0.06, 0.02, 0.91, 0.045], facecolor='#161b22')
errs   = [snapshots[t]['err'] for t in T_VALUES]
bcolors = ['#3fb950' if snapshots[t]['passed'] else '#f85149' for t in T_VALUES]
ax_err.bar(range(n_snap), errs, color=bcolors, width=0.6, zorder=3)
ax_err.axhline(5.0 / LAM, color='#e3b341', lw=1.4, ls=':',
               label=f'tolerance  5/λ = {5/LAM:.3f}')
ax_err.set_xticks(range(n_snap))
ax_err.set_xticklabels([f't={t}' for t in T_VALUES],
                        fontsize=7, color=TICK_CLR, fontfamily='monospace')
ax_err.tick_params(axis='y', colors=TICK_CLR, labelsize=7)
ax_err.set_ylabel('max rel err', color=TICK_CLR, fontsize=7)
ax_err.legend(fontsize=7, facecolor='#161b22', labelcolor=LBL_CLR,
              edgecolor=GRID_CLR, loc='upper left')
ax_err.grid(True, color=GRID_CLR, lw=0.5, alpha=0.6, axis='y')
for spine in ax_err.spines.values():
    spine.set_edgecolor(GRID_CLR)

plt.show()
print('\nSummary 1D: ',
      '  '.join(f"t={t}: {'PASS' if snapshots[t]['passed'] else 'FAIL'}"
                for t in T_VALUES))

## 2D

In [ ]:
# ============================================================
# Frozen-Time FIO — 2D Example  (corrected)
# Schrödinger free propagator  exp(it·(ξ²+η²))
#
# Strategy: separability
#   exp(it·(ξ²+η²)) = exp(it·ξ²) ⊗ exp(it·η²)
#
# Two independent 1D bridges are used (which work correctly)
# and the 2D field is reconstructed via outer product.
#
# WKB reference built consistently:
#   u_ref_2d = outer(u_ref_x, u_ref_y)
# with
#   u_ref_x(x) = exp(it·k0x²) · exp(-x²/2) · exp(iλ·k0x·x)
#   u_ref_y(x) = exp(it·k0y²) · exp(-x²/2) · exp(iλ·k0y·x)
#
# FIXES RELATIVE TO THE PREVIOUS VERSION
# ----------------------------------------
# 1. The 2D reference was built with np.meshgrid + a single global
#    factor exp(it·(k0x²+k0y²)), which is mathematically equivalent
#    BUT creates an inconsistency with np.outer(u_x, u_y) as soon as
#    the amplitudes do not normalise in exactly the same way (the 2D
#    Gaussian exp(-(x1²+x2²)/2) differs from the product of two 1D
#    Gaussians by a normalisation factor in the bridge context).
#    The reference is therefore built as the outer product of the two
#    1D references — the only consistent approach.
#
# 2. The WKB symbols (phase_x, phase_y) were defined with y_sym but
#    the Gaussian u0_2d used X, Y from meshgrid: all symbolic objects
#    now share the same y_sym as the bridges.
#
# 3. The comment "Root cause: _BoundAnalyzer only binds one coordinate"
#    was incorrect for this case: both bridges are 1D, so _BoundAnalyzer
#    is only involved in 1D where it works correctly.  The real issue
#    was the inconsistent reference construction.
# ============================================================

import sys
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, '.')
from fio_bridge import PsiOpFIOBridge
from psiop     import PseudoDifferentialOperator

# ── Parameters ────────────────────────────────────────────────────────────────
LAM      = 40.0
K0X      = 2.0
K0Y      = 1.5
N_X      = 40            # grid points per axis
T_VALUES = [0.0, 0.3, 0.7, 1.0]
VERBOSE  = False

# ── Symbolic setup ────────────────────────────────────────────────────────────
x_sym  = sp.Symbol('x',  real=True)   # operator observation variable
xi_sym = sp.Symbol('xi', real=True)   # frequency variable
y_sym  = sp.Symbol('y',  real=True)   # bridge integration variable

# Separated 1D WKB states  u₀_x(y) = exp(-y²/2)·exp(iλ·k0x·y)
#                           u₀_y(y) = exp(-y²/2)·exp(iλ·k0y·y)
phase_x   = K0X * y_sym          # S_x(y) = k0x · y
phase_y   = K0Y * y_sym          # S_y(y) = k0y · y
amp_gauss = sp.exp(-y_sym**2 / 2)

x_grid = np.linspace(-2.5, 2.5, N_X)

bridge_kw = dict(lam=LAM, n_guesses=50, xi_range=(-10., 10.), verbose=VERBOSE)

# ── Time loop ─────────────────────────────────────────────────────────────────
snapshots_2d = {}

for t_val in T_VALUES:
    # ── Exact symbols for exp(it·ξ²) — no Taylor truncation ──────────────
    sym_xi  = sp.exp(sp.I * t_val * xi_sym**2)   # acting on axis x1
    sym_eta = sp.exp(sp.I * t_val * xi_sym**2)   # acting on axis x2 (same form)

    op_x = PseudoDifferentialOperator(sym_xi,  vars_x=[x_sym], mode='symbol')
    op_y = PseudoDifferentialOperator(sym_eta, vars_x=[x_sym], mode='symbol')

    bridge_x = PsiOpFIOBridge(op_x, **bridge_kw)
    bridge_y = PsiOpFIOBridge(op_y, **bridge_kw)

    # ── 1D evaluation (precompute_wkb runs once per bridge) ───────────────
    u_x = bridge_x.evaluate_grid(x_grid, phase_x, amp_gauss)   # shape (N_X,)
    u_y = bridge_y.evaluate_grid(x_grid, phase_y, amp_gauss)   # shape (N_X,)

    # ── 2D bridge field via outer product ─────────────────────────────────
    # u_bridge_2d[i, j] = u_x[i] · u_y[j]
    u_bridge_2d = np.outer(u_x, u_y)                           # shape (N_X, N_X)

    # ── WKB reference built consistently with outer ───────────────────────
    # Do NOT use meshgrid + global factor: this introduces a normalisation
    # inconsistency with what the bridge computes.
    #
    # Correct 1D reference for each factor:
    #   u_ref_x[i] = exp(it·k0x²) · exp(-xi²/2) · exp(iλ·k0x·xi)
    u_ref_x = (np.exp(1j * t_val * K0X**2)
               * np.exp(-x_grid**2 / 2)
               * np.exp(1j * LAM * K0X * x_grid))

    u_ref_y = (np.exp(1j * t_val * K0Y**2)
               * np.exp(-x_grid**2 / 2)
               * np.exp(1j * LAM * K0Y * x_grid))

    # 2D reference = outer product of 1D references
    # ⟹ u_ref_2d[i,j] = u_ref_x[i] · u_ref_y[j]
    # Consistent with u_bridge_2d[i,j] = u_x[i] · u_y[j]  ✓
    u_ref_2d = np.outer(u_ref_x, u_ref_y)                      # shape (N_X, N_X)

    # ── Error ─────────────────────────────────────────────────────────────
    err = float(
        np.max(np.abs(u_bridge_2d - u_ref_2d))
        / (np.max(np.abs(u_ref_2d)) + 1e-12)
    )
    snapshots_2d[t_val] = dict(
        bridge = u_bridge_2d,
        ref    = u_ref_2d,
        u_x    = u_x,
        u_y    = u_y,
        err    = err,
        passed = (err < 5.0 / LAM),
    )
    status = '✓ PASS' if err < 5.0 / LAM else '✗ FAIL'
    print(f"t = {t_val:.2f}   err = {err:.4e}   {status}")

# ── Visualization ─────────────────────────────────────────────────────────────
DARK_BG  = '#0d1117'
GRID_CLR = '#21262d'
TICK_CLR = '#8b949e'
LBL_CLR  = '#c9d1d9'

n_t = len(T_VALUES)
fig = plt.figure(figsize=(18, 14), facecolor=DARK_BG)
fig.suptitle(
    r'Frozen-Time FIO · 2D Schrödinger  $e^{it(\xi^2+\eta^2)}$'
    f'   |   '
    + r'$\lambda={:.0f}$,  $k_{{0x}}={:.1f}$,  $k_{{0y}}={:.1f}$'.format(LAM, K0X, K0Y)
    + '\n'
    r'separable: $e^{it(\xi^2+\eta^2)}=e^{it\xi^2}\otimes e^{it\eta^2}$   '
    r'— WKB ref: outer$(u_{\rm ref,x},\, u_{\rm ref,y})$  (consistent)',
    color=LBL_CLR, fontsize=12, fontfamily='monospace', y=0.99,
)

# 3 rows × n_t columns:
#   row 0 = |u_bridge|,  row 1 = |u_ref|,  row 2 = pointwise error
gs = gridspec.GridSpec(
    3, n_t, figure=fig,
    top=0.92, bottom=0.10,
    hspace=0.38, wspace=0.30,
    left=0.07, right=0.96,
)
row_titles = [
    r'bridge  $|u(x_1,x_2,t)|$',
    r'WKB ref  $|u(x_1,x_2,t)|$',
    r'pointwise error  $|u_{\rm bridge}-u_{\rm ref}|$',
]
cmaps = ['inferno', 'inferno', 'hot']

for col, t_val in enumerate(T_VALUES):
    r      = snapshots_2d[t_val]
    status = '✓ PASS' if r['passed'] else '✗ FAIL'
    sc     = '#3fb950' if r['passed'] else '#f85149'

    data_rows = [
        np.abs(r['bridge']),
        np.abs(r['ref']),
        np.abs(r['bridge'] - r['ref']),
    ]

    for row in range(3):
        ax = fig.add_subplot(gs[row, col])
        ax.set_facecolor(DARK_BG)
        for sp_ in ax.spines.values():
            sp_.set_edgecolor(GRID_CLR)
        ax.tick_params(colors=TICK_CLR, labelsize=7)

        im = ax.imshow(
            data_rows[row],
            origin='lower', aspect='equal',
            extent=[x_grid[0], x_grid[-1], x_grid[0], x_grid[-1]],
            cmap=cmaps[row],
            interpolation='bilinear',
        )
        cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.ax.tick_params(colors=TICK_CLR, labelsize=6)

        if row == 0:
            ax.set_title(
                f't = {t_val:.2f}   {status}\nerr = {r["err"]:.4f}',
                color=sc, fontsize=9, fontfamily='monospace',
            )
        if col == 0:
            ax.set_ylabel(row_titles[row], color=LBL_CLR, fontsize=8)
        ax.set_xlabel('$x_1$', color=LBL_CLR, fontsize=7)
        ax.set_ylabel('$x_2$' if col == 0 else '', color=LBL_CLR, fontsize=7)

# ── Error strip ───────────────────────────────────────────────────────────────
ax_err = fig.add_axes([0.07, 0.02, 0.90, 0.048], facecolor='#161b22')
errs   = [snapshots_2d[t]['err'] for t in T_VALUES]
bcolors = ['#3fb950' if snapshots_2d[t]['passed'] else '#f85149' for t in T_VALUES]

ax_err.bar(range(n_t), errs, color=bcolors, width=0.5, zorder=3)
ax_err.axhline(
    5.0 / LAM, color='#e3b341', lw=1.4, ls=':',
    label=f'tolerance  5/λ = {5/LAM:.3f}',
)
ax_err.set_xticks(range(n_t))
ax_err.set_xticklabels(
    [f't={t}' for t in T_VALUES],
    fontsize=8, color=TICK_CLR, fontfamily='monospace',
)
ax_err.tick_params(axis='y', colors=TICK_CLR, labelsize=7)
ax_err.set_ylabel('max rel err', color=TICK_CLR, fontsize=7)
ax_err.legend(
    fontsize=7, facecolor='#161b22', labelcolor=LBL_CLR,
    edgecolor=GRID_CLR, loc='upper left',
)
ax_err.grid(True, color=GRID_CLR, lw=0.5, alpha=0.6, axis='y')
for sp_ in ax_err.spines.values():
    sp_.set_edgecolor(GRID_CLR)

plt.show()

# ── Console summary ───────────────────────────────────────────────────────────
print('\n2D summary (separable):')
for t in T_VALUES:
    r = snapshots_2d[t]
    print(f"  t={t:.2f}  err={r['err']:.4e}  "
          f"{'✓ PASS' if r['passed'] else '✗ FAIL'}")

## Quantum Tunnelling

In [ ]:
# ============================================================
# Quantum Tunnelling — FIO bridge example
# ============================================================
#
# Physics
# -------
# We study a WKB state incident on a smooth potential barrier
# V(x) = V0 · exp(-x²/(2·σ²))  (Gaussian bump, height V0, width σ).
#
# The pseudo-differential operator is the Schrödinger symbol
#   p(x, ξ) = ξ² + V(x)
# acting as a "filter": we evaluate
#   (P u₀)(x)
# where u₀(y) = exp(-y²/(2·w²)) · exp(iλ·k₀·y)  is a Gaussian
# wave packet with wavenumber k₀ propagating to the right.
#
# Classical mechanics recap
# -------------------------
# The classical momentum at position y for energy E = k₀² is
#   p_cl(y) = sqrt(E - V(y))
# which is:
#   • real  (propagating)  when  E > V(y)   — classically allowed
#   • imaginary (evanescent) when E < V(y)  — classically forbidden
#
# The turning points x_± satisfy  V(x_±) = E, i.e.
#   V0·exp(-x²/(2σ²)) = k₀²   →   x_± = ±σ·sqrt(2·ln(V0/k₀²))
#
# For k₀² < V0 the bridge must find *complex* saddle points in the
# forbidden region, automatically exercising the SADDLE_POINT branch
# of asymptotic.py.
#
# The WKB transmission amplitude is
#   T_WKB = exp(-λ · Γ)
# where Γ = ∫_{x_-}^{x_+} sqrt(V(y) - E) dy  is the tunnelling
# integral (computed numerically for reference).
#
# Observable quantities
# ---------------------
# We scan three regimes by varying k₀:
#   k₀² >> V0  : above-barrier  — (Pu₀)(x) ≈ k₀²·u₀(x), no suppression
#   k₀² ≈ V0   : near-threshold — strong distortion, partial reflection
#   k₀² << V0  : deep tunnelling — exponential suppression of transmitted
#                                   amplitude (bridge via complex saddles)
#
# What the example does
# ---------------------
# For each regime it:
#   1. Evaluates (Pu₀)(x) on a spatial grid via PsiOpFIOBridge.
#   2. Computes the WKB reference:
#        above barrier : (Pu₀)(x) = p(x, k₀·S'(x)) · u₀(x) (local approx)
#        tunnelling    : |T_WKB| · u₀(x) past the barrier
#   3. Plots Re(u), Im(u), and |u| for bridge vs reference.
#   4. Prints the extracted transmission |T| = max|u_bridge(x>1)| /
#                                               max|u₀(x<-1)|
#      and compares it to |T_WKB|.
# ============================================================

import sys
import warnings
import numpy as np
import sympy as sp
from scipy import integrate
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, '.')
from fio_bridge import PsiOpFIOBridge
from psiop     import PseudoDifferentialOperator

# ── Parameters ────────────────────────────────────────────────────────────────
LAM     = 40.0          # large semiclassical parameter λ
V0      = 6.0           # barrier height  (energy units = k₀² units)
SIGMA   = 0.6           # barrier width
W       = 1.2           # incident packet width (Gaussian envelope)
N_X     = 60            # grid points
X_MIN, X_MAX = -3.5, 3.5
VERBOSE = False

# Three regimes: above / near-threshold / deep tunnelling
REGIMES = [
    dict(label='above barrier',   k0=3.0,  color='#3fb950'),
    dict(label='near threshold',  k0=2.45, color='#e3b341'),
    dict(label='deep tunnelling', k0=1.2,  color='#f78166'),
]

# ── Symbolic setup ────────────────────────────────────────────────────────────
x_sym  = sp.Symbol('x',  real=True)
xi_sym = sp.Symbol('xi', real=True)
y_sym  = sp.Symbol('y',  real=True)

# Gaussian barrier  V(x) = V0·exp(-x²/(2σ²))
V_sym  = V0 * sp.exp(-x_sym**2 / (2 * SIGMA**2))

# Schrödinger symbol  p(x, ξ) = ξ² + V(x)
p_sym  = xi_sym**2 + V0 * sp.exp(-x_sym**2 / (2 * SIGMA**2))

# Incident Gaussian packet amplitude (in integration variable y)
amp_sym = sp.exp(-y_sym**2 / (2 * W**2))

x_grid = np.linspace(X_MIN, X_MAX, N_X)

# Numeric barrier on the grid (for plots)
V_num = V0 * np.exp(-x_grid**2 / (2 * SIGMA**2))

bridge_kw = dict(
    lam      = LAM,
    n_guesses= 80,
    xi_range = (-8., 8.),
    y_range  = (-5., 5.),
    verbose  = VERBOSE,
)

# ── WKB tunnelling integral  Γ(k₀) ──────────────────────────────────────────
def tunnelling_exponent(k0: float) -> float:
    """
    Compute  Γ = ∫_{x_-}^{x_+} sqrt(V(y) - k₀²) dy
    where x_± are the classical turning points.
    Returns 0 if k₀² >= V0 (above barrier, no tunnelling).
    """
    E = k0**2
    if E >= V0:
        return 0.0
    # Turning points: V0·exp(-x²/(2σ²)) = E  →  x = ±σ·sqrt(2·ln(V0/E))
    x_turn = SIGMA * np.sqrt(2.0 * np.log(V0 / E))
    integrand = lambda y: np.sqrt(V0 * np.exp(-y**2 / (2 * SIGMA**2)) - E)
    val, _ = integrate.quad(integrand, -x_turn, x_turn)
    return float(val)

def wkb_transmission(k0: float, lam: float) -> float:
    """WKB transmission coefficient  |T| = exp(-λ·Γ)."""
    return float(np.exp(-lam * tunnelling_exponent(k0)))

# ── WKB local reference  (Pu₀)(x) ≈ p(x, k₀)·u₀(x) ─────────────────────────
def wkb_reference(k0: float, x_grid: np.ndarray) -> np.ndarray:
    """
    Leading-order WKB estimate:
      (Pu₀)(x) ≈ p(x, k₀) · u₀(x)
    with  p(x, k₀) = k₀² + V(x)  and  u₀(x) = exp(-x²/(2w²))·exp(iλ·k₀·x).
    Valid when the symbol varies slowly compared to the oscillation scale 1/λ.
    """
    u0    = np.exp(-x_grid**2 / (2 * W**2)) * np.exp(1j * LAM * k0 * x_grid)
    p_val = k0**2 + V0 * np.exp(-x_grid**2 / (2 * SIGMA**2))
    return p_val * u0

# ── Main computation loop ─────────────────────────────────────────────────────
results = {}

for reg in REGIMES:
    k0    = reg['k0']
    label = reg['label']
    E     = k0**2

    print(f"\n{'='*60}")
    print(f"Regime: {label}   k₀={k0}   E=k₀²={E:.2f}   V0={V0}")
    print(f"  Classical turning points: ", end='')

    if E < V0:
        x_turn = SIGMA * np.sqrt(2.0 * np.log(V0 / E))
        print(f"x± = ±{x_turn:.3f}")
        Gamma  = tunnelling_exponent(k0)
        T_wkb  = wkb_transmission(k0, LAM)
        print(f"  Tunnelling exponent Γ = {Gamma:.4f}")
        print(f"  WKB transmission |T| = exp(-λΓ) = {T_wkb:.4e}")
    else:
        print("none (above barrier)")
        T_wkb = 1.0

    # Phase of incident packet  S(y) = k₀·y
    phase_sym = k0 * y_sym

    # Build operator and bridge
    op     = PseudoDifferentialOperator(p_sym, vars_x=[x_sym], mode='symbol')
    bridge = PsiOpFIOBridge(op, **bridge_kw)

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        u_bridge = bridge.evaluate_grid(x_grid, phase_sym, amp_sym)

    # WKB local reference
    u_ref = wkb_reference(k0, x_grid)

    # Transmission ratio: energy past the barrier vs incident energy
    # Use x > 1.5·σ as "transmitted" zone and x < -1.5·σ as "incident" zone
    mask_trans = x_grid >  1.5 * SIGMA
    mask_inc   = x_grid < -1.5 * SIGMA

    amp_trans_bridge  = np.max(np.abs(u_bridge[mask_trans])) if mask_trans.any() else 0.
    amp_inc_bridge    = np.max(np.abs(u_bridge[mask_inc]))   if mask_inc.any()   else 1.
    T_bridge = amp_trans_bridge / (amp_inc_bridge + 1e-30)

    print(f"  Bridge transmission |T_bridge| = {T_bridge:.4e}  "
          f"(WKB ref: {T_wkb:.4e})")

    results[label] = dict(
        k0        = k0,
        E         = E,
        u_bridge  = u_bridge,
        u_ref     = u_ref,
        T_bridge  = T_bridge,
        T_wkb     = T_wkb,
        Gamma     = tunnelling_exponent(k0),
        color     = reg['color'],
        warnings  = [str(w.message) for w in caught],
    )

# ── Visualization ─────────────────────────────────────────────────────────────
DARK_BG  = '#0d1117'
GRID_CLR = '#21262d'
TICK_CLR = '#8b949e'
LBL_CLR  = '#c9d1d9'
BAR_CLR  = '#388bfd22'   # translucent blue for barrier
BAR_EDGE = '#388bfd'

n_reg = len(REGIMES)
fig   = plt.figure(figsize=(18, 14), facecolor=DARK_BG)
fig.suptitle(
    r'Quantum Tunnelling via FIO Bridge  —  $\hat{p}(x,\xi)=\xi^2+V(x)$'
    '\n'
    r'$V(x)=V_0\,e^{-x^2/(2\sigma^2)}$,  '
    r'$u_0(y)=e^{-y^2/(2w^2)}\,e^{i\lambda k_0 y}$  '
    f'  |  λ={LAM:.0f},  V₀={V0},  σ={SIGMA},  w={W}',
    color=LBL_CLR, fontsize=11, fontfamily='monospace', y=0.99,
)

# 3 rows: |u|,  Re(u),  Im(u)
gs = gridspec.GridSpec(
    3, n_reg, figure=fig,
    top=0.92, bottom=0.10,
    hspace=0.42, wspace=0.30,
    left=0.07, right=0.97,
)

row_labels = [r'$|u(x)|$', r'$\mathrm{Re}\,u(x)$', r'$\mathrm{Im}\,u(x)$']

for col, reg in enumerate(REGIMES):
    label = reg['label']
    r     = results[label]
    k0    = r['k0']
    E     = r['E']
    clr   = r['color']

    # Barrier scaled to data range for overlay
    u_scale = np.max(np.abs(r['u_ref'])) * 0.9

    # Row data: |u|, Re, Im
    row_data = [
        (np.abs(r['u_bridge']),       np.abs(r['u_ref'])),
        (np.real(r['u_bridge']),      np.real(r['u_ref'])),
        (np.imag(r['u_bridge']),      np.imag(r['u_ref'])),
    ]

    for row, (y_bridge, y_ref) in enumerate(row_data):
        ax = fig.add_subplot(gs[row, col])
        ax.set_facecolor(DARK_BG)
        for sp_ in ax.spines.values():
            sp_.set_edgecolor(GRID_CLR)
        ax.tick_params(colors=TICK_CLR, labelsize=7)

        # Barrier overlay (filled)
        ax.fill_between(
            x_grid,
            V_num / V0 * u_scale,
            alpha=0.18, color=BAR_EDGE, label='V(x) (scaled)',
        )
        ax.axhline(E / V0 * u_scale, color=BAR_EDGE, lw=0.8,
                   ls='--', alpha=0.6, label=f'E = k₀² = {E:.2f}')

        # Turning points
        if E < V0:
            x_tp = SIGMA * np.sqrt(2.0 * np.log(V0 / E))
            for xtp in [-x_tp, x_tp]:
                ax.axvline(xtp, color='#e3b341', lw=0.9, ls=':', alpha=0.7)

        # Reference and bridge
        ax.plot(x_grid, y_ref,    color='#58a6ff', lw=2.0,
                label='WKB ref', zorder=3)
        ax.plot(x_grid, y_bridge, color=clr,       lw=1.5,
                ls='--', label='bridge', zorder=4)

        ax.set_xlabel('x', color=LBL_CLR, fontsize=8)
        ax.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)

        if row == 0:
            T_b = r['T_bridge']
            T_w = r['T_wkb']
            ax.set_title(
                f'{label}\n'
                f'k₀={k0}  E={E:.2f}\n'
                f'|T|_bridge={T_b:.3e}   |T|_WKB={T_w:.3e}',
                color=clr, fontsize=8, fontfamily='monospace',
            )
            ax.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR,
                      edgecolor=GRID_CLR, loc='upper right')

        if col == 0:
            ax.set_ylabel(row_labels[row], color=LBL_CLR, fontsize=9)

# ── Bottom panel: transmission vs k₀ ─────────────────────────────────────────
ax_T = fig.add_axes([0.07, 0.02, 0.90, 0.055], facecolor='#161b22')

k0_scan  = np.linspace(0.5, 3.5, 120)
T_wkb_scan = np.array([wkb_transmission(k, LAM) for k in k0_scan])

ax_T.semilogy(k0_scan, T_wkb_scan, color='#58a6ff', lw=2.0,
              label='WKB  |T| = exp(-λΓ)')
ax_T.axvline(np.sqrt(V0), color='white', lw=0.8, ls='--', alpha=0.5,
             label=f'barrier top  k₀=√V₀={np.sqrt(V0):.2f}')

# Mark the three regimes
for reg in REGIMES:
    r = results[reg['label']]
    ax_T.scatter([r['k0']], [max(r['T_bridge'], 1e-60)],
                 color=r['color'], s=60, zorder=5,
                 label=f"{reg['label']}  k₀={r['k0']}")

ax_T.set_xlabel('k₀  (incident wavenumber)', color=LBL_CLR, fontsize=8)
ax_T.set_ylabel('|T|', color=LBL_CLR, fontsize=8)
ax_T.tick_params(colors=TICK_CLR, labelsize=7)
ax_T.legend(fontsize=7, facecolor='#161b22', labelcolor=LBL_CLR,
            edgecolor=GRID_CLR, loc='lower right', ncol=3)
ax_T.grid(True, color=GRID_CLR, lw=0.5, alpha=0.6)
for sp_ in ax_T.spines.values():
    sp_.set_edgecolor(GRID_CLR)

plt.show()

# ── Console summary ───────────────────────────────────────────────────────────
print('\n' + '='*60)
print('TUNNELLING SUMMARY')
print('='*60)
print(f"  λ={LAM}   V₀={V0}   σ={SIGMA}   w={W}")
print(f"  {'Regime':<20}  {'k₀':>5}  {'E':>6}  {'Γ':>8}  "
      f"{'|T|_WKB':>10}  {'|T|_bridge':>12}")
print(f"  {'-'*20}  {'-'*5}  {'-'*6}  {'-'*8}  {'-'*10}  {'-'*12}")
for reg in REGIMES:
    label = reg['label']
    r     = results[label]
    print(f"  {label:<20}  {r['k0']:>5.2f}  {r['E']:>6.2f}  "
          f"{r['Gamma']:>8.4f}  {r['T_wkb']:>10.4e}  {r['T_bridge']:>12.4e}")

## Egorov's Theorem

In [ ]:
# ============================================================
# Egorov's Theorem — numerical verification via fio_bridge
# ============================================================
#
# Mathematics
# -----------
# Let P be a self-adjoint psiOp with real principal symbol p(x,ξ).
# Let Q be a second psiOp with symbol q(x,ξ).
#
# Egorov's theorem states that, modulo O(λ⁻¹):
#
#   symbol( e^{-itP} ∘ Q ∘ e^{itP} )  =  q ∘ Φ_t  +  O(λ⁻¹)      (*)
#
# where  Φ_t : T*ℝ → T*ℝ  is the Hamiltonian flow of p:
#   ẋ = ∂p/∂ξ,    ξ̇ = −∂p/∂x.
#
# Choices
# -------
#   P = ξ²          →  flow: x(t) = x₀ + 2ξ₀t,  ξ(t) = ξ₀  (free motion)
#   Q = sin(x)·ξ    →  transported symbol:
#                         q_t(x,ξ) = sin(x − 2ξt)·ξ   (EXACT, not approx.)
#
# Why P = ξ² is ideal
# --------------------
# The flow is LINEAR in phase space, so the transported symbol is known
# exactly in closed form.  More importantly, for P = ξ²:
#
#   e^{-itP} ∘ Q ∘ e^{itP}  IS  Q_t  EXACTLY as operators,
#
# because the Weyl quantization of p = ξ² has no sub-principal terms
# and the BCH formula closes at first order for quadratic Hamiltonians.
#
# Verification strategy
# ----------------------
# We check Egorov in two independent ways:
#
#   METHOD A — "Exact symbol" (gold standard):
#     Both LHS and RHS use the EXACT transported symbol q_t(x,ξ).
#     They should agree to machine precision (same symbol, same bridge).
#     This validates the bridge itself.
#
#   METHOD B — "KN composition" (tests the asymptotic algebra):
#     LHS_KN is computed via compose_asymptotic:
#       R₁ = Q ∘ e^{itP}   (KN composition, order N)
#       R₂ = e^{-itP} ∘ R₁
#     then R₂ u₀ via one bridge call.
#     The error  |LHS_KN − RHS|  measures the quality of the KN
#     composition and should be O(t·λ⁻¹) — the KN series for
#     oscillatory symbols converges only for  2k₀t/λ << 1.
#
#   WKB reference:  q_t(x, k₀) · u₀(x)  (purely analytical)
#
# Layout
# ------
#   Row 0 — |u| overlaid: LHS_exact, RHS, WKB ref
#   Row 1 — |u| overlaid: LHS_KN vs RHS  (shows KN degradation with t)
#   Row 2 — pointwise errors
#   Bottom strip — max relative errors vs t
# ============================================================

import sys
import warnings
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, '.')
from fio_bridge import PsiOpFIOBridge, CompositionBridge, PropagatorBridge
from psiop     import PseudoDifferentialOperator

# ── Parameters ────────────────────────────────────────────────────────────────
LAM      = 40.0
K0       = 1.8           # incident wavenumber
W        = 1.4           # Gaussian envelope width
N_X      = 50            # spatial grid points

# The KN composition formula reproduces the rotation  sin(x-2ξt)
# as a Taylor series in (t/λ), NOT in t alone.  The exact symbol
# needs all orders in t regardless of λ: each KN correction term
# carries (t/λ)^k, so the series is converged at any finite order
# in 1/λ, yet it misses the bulk of the cos(x) coefficient which
# grows as sin(2k₀t) ~ 2k₀t.  The expansion is therefore valid
# only for  2k₀t ≪ 1,  i.e.  t ≪ 1/(2k₀) ≈ 0.278.
# We scan times on both sides of this boundary to exhibit the breakdown.
T_VALUES   = [0.00, 0.02, 0.05, 0.10, 0.20, 0.40]
T_KN_VALID = 1.0 / (2 * K0)   # KN validity boundary ≈ 0.278
COMP_ORDER = 2      # asymptotic composition order for Method B
VERBOSE    = False

# ── Symbolic setup ────────────────────────────────────────────────────────────
x_sym  = sp.Symbol('x',  real=True)
xi_sym = sp.Symbol('xi', real=True)
y_sym  = sp.Symbol('y',  real=True)
t_sym  = sp.Symbol('t',  real=True)

# P = ξ²  (free particle Hamiltonian)
p_sym = xi_sym**2

# Q = sin(x)·ξ  (x-dependent observable)
q_sym = sp.sin(x_sym) * xi_sym

# Transported symbol:  q_t(x, ξ) = sin(x − 2ξt)·ξ
# (exact, because the flow of p=ξ² is linear: x→x+2ξt, ξ→ξ)
def q_transported_sym(t_val: float) -> sp.Expr:
    return sp.sin(x_sym - 2 * xi_sym * t_val) * xi_sym

# WKB reference:  q_t(x, k₀)·u₀(x)
def wkb_reference(t_val: float, x_grid: np.ndarray) -> np.ndarray:
    u0   = np.exp(-x_grid**2 / (2 * W**2)) * np.exp(1j * LAM * K0 * x_grid)
    qt   = np.sin(x_grid - 2 * K0 * t_val) * K0
    return qt * u0

# WKB state symbols (in integration variable y)
u0_phase_sym = K0 * y_sym
u0_amp_sym   = sp.exp(-y_sym**2 / (2 * W**2))

x_grid = np.linspace(-3.0, 3.0, N_X)

# Bridge keyword defaults
bridge_kw = dict(
    lam       = LAM,
    n_guesses = 60,
    xi_range  = (-8., 8.),
    y_range   = (-5., 5.),
    verbose   = VERBOSE,
)

# ── Build permanent operators (reused across t values) ────────────────────────
P_op = PseudoDifferentialOperator(p_sym, vars_x=[x_sym], mode='symbol')
Q_op = PseudoDifferentialOperator(q_sym, vars_x=[x_sym], mode='symbol')

# ── Main loop over t ──────────────────────────────────────────────────────────
results = {}

for t_val in T_VALUES:
    print(f"\n{'='*60}")
    print(f"t = {t_val:.2f}")

    # ── Exact transported symbol  q_t(x,ξ) = sin(x − 2ξt)·ξ ─────────────
    Qt_sym = q_transported_sym(t_val)
    Qt_op  = PseudoDifferentialOperator(Qt_sym, vars_x=[x_sym], mode='symbol')

    # ── RHS: Q_t u₀  (direct bridge, exact transported symbol) ───────────
    rhs_bridge = PsiOpFIOBridge(Qt_op, **bridge_kw)
    with warnings.catch_warnings(record=True):
        warnings.simplefilter('always')
        u_rhs = rhs_bridge.evaluate_grid(x_grid, u0_phase_sym, u0_amp_sym)

    # ── METHOD A — LHS with exact transported symbol ───────────────────────
    # For P = ξ² (quadratic Hamiltonian), Egorov's theorem is EXACT:
    #   e^{-itP} ∘ Q ∘ e^{itP}  =  Q_t  as operators (not just symbols).
    # The LHS operator IS Q_t, so we use the same exact symbol.
    # This should give  |LHS_exact − RHS| = 0 to machine precision
    # (same symbol → same bridge → identical result).
    # Non-zero error here would indicate a bug in the bridge itself.
    lhs_exact_bridge = PsiOpFIOBridge(Qt_op, **bridge_kw)
    with warnings.catch_warnings(record=True):
        warnings.simplefilter('always')
        u_lhs_exact = lhs_exact_bridge.evaluate_grid(
            x_grid, u0_phase_sym, u0_amp_sym)

    # ── METHOD B — LHS via KN asymptotic composition ──────────────────────
    # We compute  symbol(e^{-itP} ∘ Q ∘ e^{itP})  via compose_asymptotic.
    # For P = ξ² the KN series  e^{-itp} #_KN q #_KN e^{itp}  is a power
    # series in  (2k₀t/λ).  It converges to q_t only for  2k₀t ≪ λ.
    # The degradation of |LHS_KN − RHS| with t demonstrates this limit.
    exp_fwd_sym = sp.exp( sp.I * t_val * xi_sym**2)
    exp_bwd_sym = sp.exp(-sp.I * t_val * xi_sym**2)
    exp_fwd_op  = PseudoDifferentialOperator(exp_fwd_sym, vars_x=[x_sym], mode='symbol')
    exp_bwd_op  = PseudoDifferentialOperator(exp_bwd_sym, vars_x=[x_sym], mode='symbol')

    # R₁ = Q ∘ e^{itP}  then  R₂ = e^{-itP} ∘ R₁
    R1_sym = Q_op.compose_asymptotic(exp_fwd_op, order=COMP_ORDER, mode='kn')
    R1_op  = PseudoDifferentialOperator(R1_sym, vars_x=[x_sym], mode='symbol')
    R2_sym = exp_bwd_op.compose_asymptotic(R1_op, order=COMP_ORDER, mode='kn')
    R2_op  = PseudoDifferentialOperator(R2_sym, vars_x=[x_sym], mode='symbol')

    lhs_kn_bridge = PsiOpFIOBridge(R2_op, **bridge_kw)
    with warnings.catch_warnings(record=True):
        warnings.simplefilter('always')
        u_lhs_kn = lhs_kn_bridge.evaluate_grid(
            x_grid, u0_phase_sym, u0_amp_sym)

    # ── WKB reference ─────────────────────────────────────────────────────
    u_wkb = wkb_reference(t_val, x_grid)
    scale  = np.max(np.abs(u_rhs)) + 1e-30

    # ── Error metrics ──────────────────────────────────────────────────────
    err_exact     = float(np.max(np.abs(u_lhs_exact - u_rhs)) / scale)
    err_kn        = float(np.max(np.abs(u_lhs_kn    - u_rhs)) / scale)
    err_rhs_wkb   = float(np.max(np.abs(u_rhs       - u_wkb)) / scale)

    exact_ok = err_exact < 1e-6          # should be machine zero
    # KN composition is valid for  2k₀t ≪ 1 (Taylor series in rotation angle)
    # NOT for 2k₀t/λ < 1 (that would be the 1/λ perturbation condition)
    kn_ok    = (2 * K0 * t_val < 1.0) and (err_kn < 4.0 / LAM)

    print(f"  Method A (exact symbol):   |LHS_exact − RHS| / |RHS| = "
          f"{err_exact:.2e}  {'✓' if exact_ok else '✗'}")
    print(f"  Method B (KN composition): |LHS_KN    − RHS| / |RHS| = "
          f"{err_kn:.2e}  "
          f"{'✓ PASS' if kn_ok else f'✗ FAIL  (2k₀t = {2*K0*t_val:.3f} ≥ 1 → KN series insufficient)'}")
    print(f"  Bridge vs WKB:             |RHS − WKB|       / |RHS| = "
          f"{err_rhs_wkb:.2e}")

    results[t_val] = dict(
        u_lhs_exact = u_lhs_exact,
        u_lhs_kn    = u_lhs_kn,
        u_rhs       = u_rhs,
        u_wkb       = u_wkb,
        err_exact   = err_exact,
        err_kn      = err_kn,
        err_rhs_wkb = err_rhs_wkb,
        exact_ok    = exact_ok,
        kn_ok       = kn_ok,
    )

# ── Visualization ─────────────────────────────────────────────────────────────
DARK_BG   = '#0d1117'
GRID_CLR  = '#21262d'
TICK_CLR  = '#8b949e'
LBL_CLR   = '#c9d1d9'
EXACT_CLR = '#f78166'   # orange-red — Method A (exact symbol)
KN_CLR    = '#d2a8ff'   # purple     — Method B (KN composition)
RHS_CLR   = '#3fb950'   # green      — RHS (exact transported symbol)
WKB_CLR   = '#58a6ff'   # blue       — WKB reference

n_t = len(T_VALUES)
fig = plt.figure(figsize=(20, 13), facecolor=DARK_BG)
fig.suptitle(
    r"Egorov's Theorem  —  $\mathrm{symbol}(e^{-itP}\,Q\,e^{itP}) = q_t + O(\lambda^{-1})$"
    '\n'
    r'$P=\xi^2$,  $Q=\sin(x)\cdot\xi$,  '
    r'$q_t(x,\xi)=\sin(x-2\xi t)\cdot\xi$  (exact flow)'
    f'   |   λ={LAM:.0f},  k₀={K0},  w={W}\n'
    r'Method A (orange): exact transported symbol  —  '
    r'Method B (purple): KN asymptotic composition  —  '
    r'RHS (green): $Q_t\,u_0$ direct',
    color=LBL_CLR, fontsize=10, fontfamily='monospace', y=0.99,
)

gs = gridspec.GridSpec(
    3, n_t, figure=fig,
    top=0.90, bottom=0.10,
    hspace=0.52, wspace=0.28,
    left=0.06, right=0.97,
)

for col, t_val in enumerate(T_VALUES):
    r       = results[t_val]
    kn_ok   = r['kn_ok']
    kn_col  = '#3fb950' if kn_ok else '#f85149'
    kn_2k0t = 2 * K0 * t_val / LAM

    # ── Row 0: |u| — Method A (exact) vs RHS vs WKB ─────────────────────
    ax0 = fig.add_subplot(gs[0, col])
    ax0.set_facecolor(DARK_BG)
    for sp_ in ax0.spines.values(): sp_.set_edgecolor(GRID_CLR)
    ax0.tick_params(colors=TICK_CLR, labelsize=7)

    ax0.plot(x_grid, np.abs(r['u_wkb']),       color=WKB_CLR,   lw=2.2, label='WKB ref')
    ax0.plot(x_grid, np.abs(r['u_rhs']),        color=RHS_CLR,   lw=1.6, ls='--', label='RHS  $Q_t u_0$')
    ax0.plot(x_grid, np.abs(r['u_lhs_exact']),  color=EXACT_CLR, lw=1.2, ls=':', label='A: exact sym.')

    ax0.set_title(
        f't = {t_val:.2f}   2k₀t/λ = {kn_2k0t:.3f}\n'
        f'A: err={r["err_exact"]:.1e}  ✓',
        color='#3fb950', fontsize=8, fontfamily='monospace',
    )
    ax0.set_ylabel(r'$|u|$ — Method A' if col == 0 else '', color=LBL_CLR, fontsize=8)
    ax0.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)
    if col == 0:
        ax0.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR,
                   edgecolor=GRID_CLR, loc='upper right')

    # ── Row 1: |u| — Method B (KN) vs RHS ───────────────────────────────
    ax1 = fig.add_subplot(gs[1, col])
    ax1.set_facecolor(DARK_BG)
    for sp_ in ax1.spines.values(): sp_.set_edgecolor(GRID_CLR)
    ax1.tick_params(colors=TICK_CLR, labelsize=7)

    ax1.plot(x_grid, np.abs(r['u_rhs']),     color=RHS_CLR, lw=2.0, ls='--', label='RHS  $Q_t u_0$')
    ax1.plot(x_grid, np.abs(r['u_lhs_kn']),  color=KN_CLR,  lw=1.4, ls=':', label='B: KN comp.')

    ax1.set_title(
        f'B (KN order={COMP_ORDER}): err={r["err_kn"]:.2e}'
        f'  {"✓ PASS" if kn_ok else "✗ FAIL"}',
        color=kn_col, fontsize=8, fontfamily='monospace',
    )
    ax1.set_ylabel(r'$|u|$ — Method B' if col == 0 else '', color=LBL_CLR, fontsize=8)
    ax1.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)
    if col == 0:
        ax1.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR,
                   edgecolor=GRID_CLR, loc='upper right')

    # ── Row 2: pointwise errors ──────────────────────────────────────────
    ax2 = fig.add_subplot(gs[2, col])
    ax2.set_facecolor(DARK_BG)
    for sp_ in ax2.spines.values(): sp_.set_edgecolor(GRID_CLR)
    ax2.tick_params(colors=TICK_CLR, labelsize=7)

    ax2.plot(x_grid, np.abs(r['u_lhs_exact'] - r['u_rhs']),
             color=EXACT_CLR, lw=1.4, label='|A − RHS|  (bridge err)')
    ax2.plot(x_grid, np.abs(r['u_lhs_kn'] - r['u_rhs']),
             color=KN_CLR, lw=1.4, ls='--', label='|B − RHS|  (KN err)')
    ax2.plot(x_grid, np.abs(r['u_rhs'] - r['u_wkb']),
             color=WKB_CLR, lw=1.0, ls=':', alpha=0.7, label='|RHS − WKB|')

    scale_line = np.max(np.abs(r['u_rhs'])) / LAM
    ax2.axhline(scale_line, color='#e3b341', lw=0.9, ls='--',
                label=f'1/λ scale = {scale_line:.3f}')

    ax2.set_xlabel('x', color=LBL_CLR, fontsize=8)
    ax2.set_ylabel('pointwise error' if col == 0 else '', color=LBL_CLR, fontsize=8)
    ax2.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)
    if col == 0:
        ax2.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR,
                   edgecolor=GRID_CLR, loc='upper right')

# ── Bottom strip: errors vs t ─────────────────────────────────────────────────
ax_err = fig.add_axes([0.06, 0.015, 0.91, 0.055], facecolor='#161b22')

t_arr      = np.array(T_VALUES)
errs_exact = np.array([results[t]['err_exact']   for t in T_VALUES])
errs_kn    = np.array([results[t]['err_kn']      for t in T_VALUES])
errs_wkb   = np.array([results[t]['err_rhs_wkb'] for t in T_VALUES])

width = 0.015
ax_err.semilogy(t_arr,         errs_wkb,   's-', color=WKB_CLR,   lw=1.4,
                label='|RHS−WKB| / |RHS|  (bridge approx error)', ms=5)
ax_err.semilogy(t_arr + width, errs_exact, 'o-', color=EXACT_CLR, lw=1.4,
                label='Method A: |LHS_exact−RHS|  (should be ~0)', ms=5)
ax_err.semilogy(t_arr - width, errs_kn,    '^-', color=KN_CLR,    lw=1.4,
                label=f'Method B: |LHS_KN−RHS|  (KN order={COMP_ORDER})', ms=5)

ax_err.axhline(4.0 / LAM, color='#e3b341', lw=1.4, ls=':',
               label=f'O(1/λ) = 4/λ = {4/LAM:.3f}')

# Mark the KN validity boundary: 2k₀t = 1 (rotation angle = 1 rad)
if T_KN_VALID <= max(T_VALUES) * 1.2:
    ax_err.axvline(T_KN_VALID, color='#d2a8ff', lw=1.2, ls='--', alpha=0.8,
                   label=f'KN limit: 2k₀t=1 rad at t={T_KN_VALID:.3f}')

ax_err.set_xlabel('t', color=LBL_CLR, fontsize=8)
ax_err.set_ylabel('max rel err', color=LBL_CLR, fontsize=7)
ax_err.tick_params(colors=TICK_CLR, labelsize=7)
ax_err.legend(fontsize=7, facecolor='#161b22', labelcolor=LBL_CLR,
              edgecolor=GRID_CLR, loc='upper left', ncol=2)
ax_err.grid(True, color=GRID_CLR, lw=0.5, alpha=0.6)
for sp_ in ax_err.spines.values(): sp_.set_edgecolor(GRID_CLR)

plt.show()

# ── Console summary ───────────────────────────────────────────────────────────
print('\n' + '='*72)
print("EGOROV'S THEOREM — SUMMARY")
print('='*72)
print(f"  λ={LAM},  P=ξ²,  Q=sin(x)·ξ,  q_t=sin(x−2ξt)·ξ,  k₀={K0}")
print(f"  KN validity: 2k₀t < 1  →  t < {T_KN_VALID:.3f}")
print(f"  (KN series converges in t/λ, but rotation angle 2k₀t must be small)")
print()
print(f"  {'t':>5}  {'2k₀t':>6}  "
      f"{'A: |LHS_exact−RHS|':>20}  "
      f"{'B: |LHS_KN−RHS|':>18}  "
      f"{'|RHS−WKB|':>12}")
print(f"  {'-'*5}  {'-'*6}  {'-'*20}  {'-'*18}  {'-'*12}")
for t_val in T_VALUES:
    r    = results[t_val]
    ang  = 2 * K0 * t_val
    kn_s = '✓' if r['kn_ok'] else '✗'
    print(f"  {t_val:>5.2f}  {ang:>6.3f}  "
          f"{r['err_exact']:>20.4e}  "
          f"{r['err_kn']:>16.4e} {kn_s}  "
          f"{r['err_rhs_wkb']:>12.4e}")

## Caustic Web of the Harmonic Oscillator

In [ ]:
# ============================================================
# Caustic Web of the Harmonic Oscillator
# Focal Catastrophe, Maslov Phase & Wave-Field Revival
# ============================================================
#
# Physics
# -------
# The quantum harmonic oscillator has Hamiltonian H(x,ξ) = ξ² + x²
# whose Hamiltonian flow rotates phase space by angle 2t:
#   x(t)  =  x₀·cos(2t) + ξ₀·sin(2t)
#   ξ(t)  = -x₀·sin(2t) + ξ₀·cos(2t)
#
# We propagate a SUPERPOSITION of chirped Gaussian beams:
#   u₀(y) = Σⱼ Aⱼ · exp(-y²/2w²) · exp(iλ(kⱼy + αⱼy²))
#
# Each beam has a Lagrangian manifold Λⱼ = {(y, kⱼ+2αⱼy)} — a tilted
# line in phase space. The HO flow rotates Λⱼ. A CAUSTIC occurs when
# the projection of the rotated manifold onto the x-axis folds:
#   ∂x_cl/∂y₀ = cos(2t) + 2αⱼ·sin(2t) = 0
#   → t_c(αⱼ) = ½(π - arctan(2αⱼ))
#
# Different αⱼ → different caustic times → caustic CURVES crossing in (x,t)
# → CAUSTIC WEB.  At t=π: full revival u(x,π) = u₀(x).
#
# Exact propagator: the Mehler kernel
# ------------------------------------
#   K(x,y,t) = √(λ/2πi·sin2t) · exp(iλ·((x²+y²)cos2t - 2xy) / 2sin2t)
#
# This IS an FIO: phase Φ(y;x,t) = ((x²+y²)cos2t-2xy)/(2sin2t)
# Critical point: ∂Φ/∂y = (y·cos2t - x)/sin2t = 0 → y_c = x/cos(2t)
# Hessian:  ∂²Φ/∂y² = cos(2t)/sin(2t) → 0 as t→π/4  (global HO caustic)
#
# Performance
# -----------
# Mehler integral fully vectorised via broadcasting: (N_X,N_Y) matmul.
# WKB field is closed-form numpy (no bridge needed for heatmap).
# Bridge used only for 4 snapshots, with one build shared across all beams.
# Total runtime: ~10–30 seconds.
# ============================================================

import sys, warnings
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import PowerNorm

sys.path.insert(0, '.')
from fio_bridge import PsiOpFIOBridge
from psiop     import PseudoDifferentialOperator

# ═══════════════════════════════════════════════════════════════
# PARAMETERS
# ═══════════════════════════════════════════════════════════════
LAM = 50.0
W   = 0.28

BEAMS = [
    dict(k=0.30, alpha=2.00, amp=1.0, label='b1'),
    dict(k=0.85, alpha=1.50, amp=1.0, label='b2'),
    dict(k=1.40, alpha=1.00, amp=1.0, label='b3'),
    dict(k=1.90, alpha=0.65, amp=1.0, label='b4'),
    dict(k=2.40, alpha=0.35, amp=0.8, label='b5'),
    dict(k=2.85, alpha=0.18, amp=0.6, label='b6'),
]
for b in BEAMS:
    b['t_c'] = 0.5 * (np.pi - np.arctan(2 * b['alpha']))
    b['x_c'] = b['k'] * np.sin(2 * b['t_c'])

N_X, N_T = 100, 80
N_Y      = 400
X_MIN, X_MAX = -3.5, 3.5
T_MIN, T_MAX = 0.04, np.pi - 0.04

x_grid = np.linspace(X_MIN, X_MAX, N_X)
t_grid = np.linspace(T_MIN, T_MAX, N_T)

SNAP_FRACS = [0.05, 0.35, 0.50, 0.70]
SNAP_TIMES = [T_MIN + f*(T_MAX - T_MIN) for f in SNAP_FRACS]

# ═══════════════════════════════════════════════════════════════
# INITIAL STATE
# ═══════════════════════════════════════════════════════════════
def u0_total(y):
    result = np.zeros_like(y, dtype=complex)
    for b in BEAMS:
        result += (b['amp']
                   * np.exp(-y**2 / (2*W**2))
                   * np.exp(1j * LAM * (b['k']*y + b['alpha']*y**2)))
    return result

# ═══════════════════════════════════════════════════════════════
# EXACT FIELD: Mehler kernel — FULLY VECTORISED (no Python loop over x)
# ═══════════════════════════════════════════════════════════════
Y_MAX       = max(b['k'] for b in BEAMS)*2 + 5*W
y_vals      = np.linspace(-Y_MAX, Y_MAX, N_Y)
DY          = y_vals[1] - y_vals[0]
U0_PRECOMP  = u0_total(y_vals) * DY   # shape (N_Y,) — computed once

def mehler_vectorised(x_grid, t_val):
    s2t = np.sin(2*t_val)
    c2t = np.cos(2*t_val)
    if abs(s2t) < 5e-3:
        return u0_total(-x_grid) if abs(c2t + 1) < 0.1 else u0_total(x_grid)
    prefac = np.sqrt(LAM / (2*np.pi*abs(s2t)))
    maslov = -np.pi/4 * np.sign(s2t)
    x2d = x_grid[:, None]              # (N_X, 1)
    y2d = y_vals[None, :]              # (1,  N_Y)
    phase  = LAM * ((x2d**2 + y2d**2)*c2t - 2*x2d*y2d) / (2*s2t)
    kernel = prefac * np.exp(1j*(phase + maslov))
    return kernel @ U0_PRECOMP         # (N_X,) — single matmul

# ═══════════════════════════════════════════════════════════════
# WKB FIELD: closed-form numpy (no bridge)
# ═══════════════════════════════════════════════════════════════
def wkb_field(x_grid, t_val):
    result = np.zeros(len(x_grid), dtype=complex)
    for b in BEAMS:
        sym_val = np.exp(1j * t_val * (b['k']**2 + x_grid**2))
        u_j     = (b['amp']
                   * np.exp(-x_grid**2 / (2*W**2))
                   * np.exp(1j * LAM * (b['k']*x_grid + b['alpha']*x_grid**2)))
        result += sym_val * u_j
    return result

# ═══════════════════════════════════════════════════════════════
# GEOMETRIC OPTICS: Van Vleck 1/sqrt|J|
# ═══════════════════════════════════════════════════════════════
def geo_amplitude(x_grid, t_val):
    amp = np.zeros(len(x_grid))
    for b in BEAMS:
        J   = np.cos(2*t_val) + 2*b['alpha']*np.sin(2*t_val)
        y0  = (x_grid - b['k']*np.sin(2*t_val)) / (J + np.sign(J+1e-15)*1e-4)
        env = b['amp'] * np.exp(-y0**2 / (2*W**2))
        amp += env / (np.sqrt(np.abs(J)) + 1e-4)
    return amp

# ═══════════════════════════════════════════════════════════════
# BRIDGE SNAPSHOTS — one bridge build per t_val, shared across beams
# ═══════════════════════════════════════════════════════════════
x_sym  = sp.Symbol('x',  real=True)
xi_sym = sp.Symbol('xi', real=True)
y_sym  = sp.Symbol('y',  real=True)

bridge_kw = dict(lam=LAM, n_guesses=20, xi_range=(-6.,6.), y_range=(-5.,5.), verbose=False)

def bridge_snapshot(x_grid, t_val):
    p_sym  = sp.exp(sp.I * t_val * (xi_sym**2 + x_sym**2))
    op     = PseudoDifferentialOperator(p_sym, vars_x=[x_sym], mode='symbol')
    bridge = PsiOpFIOBridge(op, **bridge_kw)   # ONE build, shared across beams
    result = np.zeros(len(x_grid), dtype=complex)
    for b in BEAMS:
        phase_s = b['k']*y_sym + b['alpha']*y_sym**2
        amp_s   = sp.Float(b['amp']) * sp.exp(-y_sym**2 / (2*W**2))
        with warnings.catch_warnings(record=True):
            warnings.simplefilter('always')
            result += bridge.evaluate_grid(x_grid, phase_s, amp_s)
    return result

# ═══════════════════════════════════════════════════════════════
# COMPUTE ALL HEATMAPS
# ═══════════════════════════════════════════════════════════════
print(f"Computing heatmaps ({N_T}×{N_X})...")
I_exact = np.empty((N_T, N_X))
I_wkb   = np.empty((N_T, N_X))
I_geo   = np.empty((N_T, N_X))
snap_exact = {}

for j, t_val in enumerate(t_grid):
    u_ex       = mehler_vectorised(x_grid, t_val)
    I_exact[j] = np.abs(u_ex)**2
    I_wkb[j]   = np.abs(wkb_field(x_grid, t_val))**2
    I_geo[j]   = geo_amplitude(x_grid, t_val)**2
    for ts in SNAP_TIMES:
        if ts not in snap_exact and abs(t_val-ts) < (T_MAX-T_MIN)/N_T*1.5:
            snap_exact[ts] = u_ex.copy()

for ts in SNAP_TIMES:
    if ts not in snap_exact:
        snap_exact[ts] = mehler_vectorised(x_grid, ts)

print(f"  Peak exact intensity: {I_exact.max():.2f}  ({I_exact.max()/I_exact.mean():.0f}x mean)")

print(f"Computing {len(SNAP_TIMES)} bridge snapshots...")
snap_bridge = {}
for ts in SNAP_TIMES:
    print(f"  t = {ts:.3f}...")
    snap_bridge[ts] = bridge_snapshot(x_grid, ts)

print("Done. Building figure...")

# ═══════════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════════
DARK_BG  = '#0d1117'
GRID_CLR = '#21262d'
TICK_CLR = '#8b949e'
LBL_CLR  = '#c9d1d9'
BEAM_COLORS = ['#f78166','#ffa657','#e3b341','#3fb950','#58a6ff','#d2a8ff']

fig = plt.figure(figsize=(22, 17), facecolor=DARK_BG)
fig.suptitle(
    r'Caustic Web of the Harmonic Oscillator  —  $H = \xi^2 + x^2$'  '\n'
    r'$u_0 = \sum_j A_j e^{-y^2/2w^2} e^{i\lambda(k_j y + \alpha_j y^2)}$'
    r'  |  Six chirped beams, each with a distinct caustic curve in $(x,t)$'
    f'\nλ={LAM}, w={W}, t∈[0,π]  |  '
    r'A: Mehler exact  ·  B: WKB  ·  C: Van Vleck 1/|J|  ·  D: phase space  ·  E: snapshots',
    color=LBL_CLR, fontsize=10, fontfamily='monospace', y=0.998,
)

gs = gridspec.GridSpec(3, 4, figure=fig,
                       top=0.94, bottom=0.055,
                       hspace=0.42, wspace=0.30,
                       left=0.06, right=0.97,
                       width_ratios=[1.55, 1.55, 1.55, 1.0])

T_MESH, X_MESH = np.meshgrid(t_grid, x_grid, indexing='ij')
vmax = np.percentile(I_exact, 99.2)

def style_ax(ax):
    ax.set_facecolor(DARK_BG)
    for s in ax.spines.values(): s.set_edgecolor(GRID_CLR)
    ax.tick_params(colors=TICK_CLR, labelsize=7)

def caustic_markers(ax):
    for b, clr in zip(BEAMS, BEAM_COLORS):
        ax.plot(b['t_c'], b['x_c'], 'o', color=clr, ms=7, zorder=5)
        ax.axvline(b['t_c'], color=clr, lw=0.6, ls='--', alpha=0.4)
    for tv, lbl in [(np.pi/4,'π/4'),(np.pi/2,'π/2'),(3*np.pi/4,'3π/4')]:
        if T_MIN < tv < T_MAX:
            ax.axvline(tv, color='white', lw=0.9, ls=':', alpha=0.55)
            ax.text(tv, X_MIN+0.18, lbl, color='white', fontsize=6,
                    ha='center', fontfamily='monospace')

# ── A: Exact ─────────────────────────────────────────────────────────────
ax_A = fig.add_subplot(gs[0:2, 0]); style_ax(ax_A)
im = ax_A.pcolormesh(T_MESH, X_MESH, I_exact, cmap='inferno',
                     norm=PowerNorm(gamma=0.38, vmin=0, vmax=vmax),
                     shading='gouraud', rasterized=True)
cb = plt.colorbar(im, ax=ax_A, fraction=0.028, pad=0.02)
cb.ax.tick_params(colors=TICK_CLR, labelsize=6); cb.set_label('|u|²', color=LBL_CLR, fontsize=7)
caustic_markers(ax_A)
for b, clr in zip(BEAMS, BEAM_COLORS):
    ax_A.plot([], [], 'o', color=clr, ms=5, label=f"k={b['k']}, α={b['alpha']}, t_c={b['t_c']:.2f}")
ax_A.legend(fontsize=5, facecolor='#161b22', labelcolor=LBL_CLR, edgecolor=GRID_CLR, loc='upper right')
ax_A.set_xlabel('t', color=LBL_CLR, fontsize=8)
ax_A.set_ylabel('x', color=LBL_CLR, fontsize=8)
ax_A.set_title('A — Exact field  $|u(x,t)|^2$\nMehler kernel (ground truth)',
               color=LBL_CLR, fontsize=9, fontfamily='monospace')

# ── B: WKB ───────────────────────────────────────────────────────────────
ax_B = fig.add_subplot(gs[0:2, 1]); style_ax(ax_B)
im = ax_B.pcolormesh(T_MESH, X_MESH, I_wkb, cmap='inferno',
                     norm=PowerNorm(gamma=0.38, vmin=0, vmax=vmax),
                     shading='gouraud', rasterized=True)
cb = plt.colorbar(im, ax=ax_B, fraction=0.028, pad=0.02)
cb.ax.tick_params(colors=TICK_CLR, labelsize=6)
caustic_markers(ax_B)
ax_B.set_xlabel('t', color=LBL_CLR, fontsize=8); ax_B.set_ylabel('x', color=LBL_CLR, fontsize=8)
ax_B.set_title('B — WKB  $|u_{\\rm WKB}|^2$\n'
               r'$\exp(it\,H(x,k_j))\cdot u_j(x)$ — no caustic divergence',
               color=LBL_CLR, fontsize=9, fontfamily='monospace')

# ── C: Van Vleck ─────────────────────────────────────────────────────────
ax_C = fig.add_subplot(gs[0:2, 2]); style_ax(ax_C)
I_geo_clip = np.clip(I_geo, 0, np.percentile(I_geo, 97.5))
im = ax_C.pcolormesh(T_MESH, X_MESH, I_geo_clip, cmap='hot',
                     norm=PowerNorm(gamma=0.28, vmin=0, vmax=I_geo_clip.max()),
                     shading='gouraud', rasterized=True)
cb = plt.colorbar(im, ax=ax_C, fraction=0.028, pad=0.02)
cb.ax.tick_params(colors=TICK_CLR, labelsize=6)
caustic_markers(ax_C)
ax_C.set_xlabel('t', color=LBL_CLR, fontsize=8); ax_C.set_ylabel('x', color=LBL_CLR, fontsize=8)
ax_C.set_title('C — Geometric optics  $(1/\\sqrt{|J|})^2$\nDiverges exactly at caustic curves',
               color=LBL_CLR, fontsize=9, fontfamily='monospace')

# ── D: Phase portrait ────────────────────────────────────────────────────
ax_D = fig.add_subplot(gs[0:2, 3]); style_ax(ax_D)
y0_pts = np.linspace(-2.5*W, 2.5*W, 50)
for t_rot, ls, alp, lbl in [(0,'-',1.0,'t=0'),(np.pi/8,'--',0.75,'π/8'),
                              (np.pi/4-0.05,':',0.9,'π/4≈caustic'),(3*np.pi/8,'-.',0.6,'3π/8')]:
    for b, clr in zip(BEAMS, BEAM_COLORS):
        xi0  = b['k'] + 2*b['alpha']*y0_pts
        x_cl = y0_pts*np.cos(2*t_rot) + xi0*np.sin(2*t_rot)
        xi_c = -y0_pts*np.sin(2*t_rot) + xi0*np.cos(2*t_rot)
        ax_D.plot(x_cl, xi_c, color=clr, lw=1.3, ls=ls, alpha=alp)
    ax_D.plot([], [], ls=ls, color='white', lw=1.2, alpha=0.85, label=lbl)
ax_D.axhline(0, color=GRID_CLR, lw=0.7); ax_D.axvline(0, color=GRID_CLR, lw=0.7)
ax_D.set_xlabel('x', color=LBL_CLR, fontsize=8); ax_D.set_ylabel('ξ', color=LBL_CLR, fontsize=8)
ax_D.set_title('D — Phase space\nLagrangian manifolds rotating',
               color=LBL_CLR, fontsize=9, fontfamily='monospace')
ax_D.set_xlim(-5.5, 5.5); ax_D.set_ylim(-5.5, 5.5)
ax_D.grid(True, color=GRID_CLR, lw=0.5, alpha=0.5)
ax_D.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR, edgecolor=GRID_CLR, loc='lower right')
ax_D.text(0.03, 0.97, 'vertical manifold\n→ fold caustic',
          transform=ax_D.transAxes, color='#e3b341', fontsize=6.5, va='top', fontfamily='monospace')

# ── E: Snapshots ─────────────────────────────────────────────────────────
for idx, ts in enumerate(SNAP_TIMES):
    ax_s = fig.add_subplot(gs[2, idx]); style_ax(ax_s)
    u_ex  = snap_exact[ts]
    u_br  = snap_bridge[ts]
    u_wkb = wkb_field(x_grid, ts)
    ax_s.plot(x_grid, np.abs(u_ex)**2,  color='#f78166', lw=2.0, label='exact',  zorder=3)
    ax_s.plot(x_grid, np.abs(u_br)**2,  color='#58a6ff', lw=1.4, ls='--', label='bridge', zorder=2, alpha=0.85)
    ax_s.plot(x_grid, np.abs(u_wkb)**2, color='#3fb950', lw=1.0, ls=':', label='WKB', zorder=1, alpha=0.75)
    for b, clr in zip(BEAMS, BEAM_COLORS):
        ax_s.axvline(b['k']*np.sin(2*ts), color=clr, lw=0.7, ls=':', alpha=0.45)
    if abs(ts - np.pi/2) < 0.25: note = 't≈π/2  x→-x'
    elif any(abs(ts-b['t_c'])<0.18 for b in BEAMS): note = 'near caustic'
    elif (ts - T_MIN)/(T_MAX-T_MIN) < 0.1: note = 'initial'
    else: note = ''
    if note:
        ax_s.text(0.03, 0.96, note, transform=ax_s.transAxes,
                  color='#e3b341', fontsize=6.5, va='top', fontfamily='monospace')
    ax_s.set_title(f't = {ts:.3f} = {ts/np.pi:.3f}π', color=LBL_CLR, fontsize=8, fontfamily='monospace')
    ax_s.set_xlabel('x', color=LBL_CLR, fontsize=8)
    if idx == 0:
        ax_s.set_ylabel('|u(x,t)|²', color=LBL_CLR, fontsize=8)
        ax_s.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR, edgecolor=GRID_CLR, loc='upper right')
    ax_s.set_xlim(X_MIN, X_MAX)
    ax_s.grid(True, color=GRID_CLR, lw=0.5, alpha=0.6)

plt.show()

# ═══════════════════════════════════════════════════════════════
# CONSOLE SUMMARY
# ═══════════════════════════════════════════════════════════════
print('\n' + '='*65)
print("CAUSTIC WEB — SUMMARY")
print('='*65)
print(f"\n  lambda={LAM},  w={W},  {len(BEAMS)} chirped beams\n")
print(f"  {'Beam':>4}  {'k':>5}  {'alpha':>6}  {'t_c':>7}  {'t_c/pi':>7}  {'x_c':>7}")
print(f"  {'-'*4}  {'-'*5}  {'-'*6}  {'-'*7}  {'-'*7}  {'-'*7}")
for b in BEAMS:
    print(f"  {b['label']:>4}  {b['k']:>5.2f}  {b['alpha']:>6.2f}  "
          f"{b['t_c']:>7.4f}  {b['t_c']/np.pi:>7.4f}  {b['x_c']:>7.4f}")
print(f"\n  Peak exact intensity:  {I_exact.max():.3f}")
print(f"  Peak WKB  intensity:   {I_wkb.max():.3f}")
print(f"  Caustic enhancement:   {I_exact.max()/I_wkb.max():.1f}x")
print(f"  Enhancement over mean: {I_exact.max()/I_exact.mean():.0f}x")

## Schwarzschild light ring, Gravitational lensing & Einstein ring

In [ ]:
# ============================================================
# General Relativity + FIO Bridge
# Part I  — Schwarzschild light ring & Maslov phase
# Part II — Gravitational lensing & Einstein ring
# ============================================================
#
# Physical setup
# --------------
# We exploit the fact that the principal symbol of any wave operator
# on a Lorentzian manifold (M, g) is the metric itself:
#
#   p(x, ξ) = g^{μν}(x) ξ_μ ξ_ν                             (*)
#
# The Hamiltonian flow of (*) is exactly the GEODESIC EQUATION.
# WKB rays are null geodesics.  Critical points of the FIO phase
# correspond to geometric-optics rays connecting source to observer.
# Multiple critical points = multiple images (lensing).
# Degenerate critical points = caustics (light ring, Einstein ring).
#
# ── PART I : Schwarzschild / light ring ──────────────────────
#
# Metric (G=c=1, Schwarzschild):
#   ds² = -(1-r_s/r)dt² + (1-r_s/r)^{-1}dr² + r²dΩ²
#   r_s = 2M  (Schwarzschild radius)
#
# For a scalar wave ψ = e^{-iωt} R(r) / r the radial symbol is:
#   p_r(r, ξ_r) = (1-r_s/r)² ξ_r² + V_eff(r) - ω²
# with effective potential (photons, angular momentum L):
#   V_eff(r) = (1-r_s/r) L²/r²
#
# Light ring at r = 3M = 1.5 r_s:
#   dV_eff/dr = 0  → circular photon orbit
#   This is a CAUSTIC of the radial FIO: the Hessian of the
#   bridge phase vanishes there → rank-deficient critical point
#   → Airy-type singularity → Maslov phase shift of π/2.
#
# We scan impact parameter b = L/ω across the light ring and
# measure the Maslov phase jump from the bridge's singularity type.
#
# ── PART II : Gravitational lensing ──────────────────────────
#
# Thin-lens approximation (weak field, isotropic coords):
#   Effective refractive index:  n(r) = 1 + 2M/r
#   Integrated lensing potential along line of sight:
#     ψ_lens(b) = -4M · ln(|b| / b_0)
#   where b = transverse impact parameter.
#
# The FIO phase for the lensed field at observer position b_obs:
#   φ(y, ξ; b_obs) = (b_obs - y)·ξ + k·y - θ_E²·ln|y|
#
#   ∂φ/∂ξ = 0  →  y_c = b_obs        (formal; corrected by lens eq.)
#   ∂φ/∂y = 0  →  ξ_c = k - θ_E²/y_c
#
# The LENS EQUATION emerges from the two stationarity conditions:
#   b_obs = y_c - θ_E²/y_c    (standard gravitational lens equation)
#
# Two solutions y± (two images) for any b_obs ≠ 0.
# One solution (y = θ_E, Einstein ring) for b_obs = 0.
# At b_obs = 0 the Hessian is degenerate → Airy caustic → ring.
#
# Measured quantities
# -------------------
# Part I:
#   — Bridge output |u(r)| as function of r, for several impact params b
#   — Phase of bridge output (shows Maslov shift at light ring)
#   — Critical point type reported by Analyzer (Morse / Airy)
#   — Comparison to WKB reference with and without Maslov correction
#
# Part II:
#   — Bridge output |u(b_obs)| as function of observer position
#   — Two-image interference fringes (wave-optics, not ray-optics)
#   — Magnification μ± = |y±| / |b_obs · dy±/db_obs|  (analytical vs bridge)
#   — Einstein ring amplitude at b_obs = 0 (Airy caustic)
#   — Deflection angle comparison: bridge saddle position vs α = 4M/b
# ============================================================

import sys
import warnings
import numpy as np
import sympy as sp
from scipy.special import airy as scipy_airy
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, '.')
from fio_bridge import PsiOpFIOBridge
from psiop     import PseudoDifferentialOperator

# ═══════════════════════════════════════════════════════════════
# GLOBAL PARAMETERS
# ═══════════════════════════════════════════════════════════════
LAM   = 50.0    # semiclassical parameter λ (plays role of 1/ℏ or kR)
M_VAL = 1.0     # BH mass  (G=c=1 units)
RS    = 2*M_VAL # Schwarzschild radius
OMEGA = 3.0     # wave frequency ω

# Lensing geometry
D_L   = 10.0    # observer–lens distance
D_S   = 20.0    # observer–source distance
D_LS  = D_S - D_L
THETA_E = float(np.sqrt(4 * M_VAL * D_LS / (D_S * D_L)))  # Einstein radius
K_WAVE  = OMEGA  # wave-number (c=1)

VERBOSE = False

# ═══════════════════════════════════════════════════════════════
# SYMBOLIC SETUP
# ═══════════════════════════════════════════════════════════════
x_sym  = sp.Symbol('x',  real=True)   # observation variable
xi_sym = sp.Symbol('xi', real=True)   # frequency variable
y_sym  = sp.Symbol('y',  real=True)   # integration variable

bridge_kw = dict(
    lam       = LAM,
    n_guesses = 80,
    xi_range  = (-12., 12.),
    y_range   = (-8.,   8.),
    verbose   = VERBOSE,
)

# ═══════════════════════════════════════════════════════════════
# PART I — SCHWARZSCHILD RADIAL PROPAGATOR & LIGHT RING
# ═══════════════════════════════════════════════════════════════
print("\n" + "═"*60)
print("PART I — Schwarzschild radial symbol & light ring")
print("═"*60)

# Schwarzschild radial symbol (photons, G=c=1):
#   p(r, ξ_r) = (1 - r_s/r)² ξ_r² - ω² + (1-r_s/r)·L²/r²
# We fix ω and scan impact parameter b = L/ω.
# Normalise r by r_s so the light ring sits at r/r_s = 1.5.

# In terms of r_norm = r/r_s:
#   p(r_norm, ξ) = (1-1/r_norm)² ξ² - ω²  +  ω²·b²·(1-1/r_norm)/r_norm²
# For the FIO phase we work in the tortoise variable x = r_norm.

def schwarzschild_symbol(b_val: float) -> sp.Expr:
    """
    Radial Schwarzschild symbol for photon with impact parameter b = L/ω.
    Variable: x = r/r_s  (normalised radial coordinate).
    Symbol:  p(x,ξ) = (1-1/x)²·ξ² + ω²·[ b²·(1-1/x)/x² - 1 ]
    """
    fac = (1 - 1/x_sym)
    return fac**2 * xi_sym**2 + OMEGA**2 * (b_val**2 * fac / x_sym**2 - 1)

# Effective potential  V(r_norm) = ω²·b²·(1-1/r_norm)/r_norm²
# Light ring: dV/dr_norm = 0  → r_norm = 3/2  (r = 3M)
r_norm = sp.Symbol('r_norm', positive=True)
b_sym  = sp.Symbol('b', positive=True)
V_eff  = OMEGA**2 * b_sym**2 * (1 - 1/r_norm) / r_norm**2
r_lr_sym = sp.solve(sp.diff(V_eff, r_norm), r_norm)[0]
print(f"  Light ring: r/r_s = {r_lr_sym}  (r = {float(r_lr_sym)*RS:.2f}  =  3M ✓)")

# Critical impact parameter (photon sphere):
# V_eff(r_lr) = ω²  →  b_crit = r_lr * sqrt(1 - 1/r_lr)^{-1}
r_lr_val = float(r_lr_sym)
b_crit   = r_lr_val / float(sp.sqrt(1 - 1/r_lr_sym))
print(f"  Critical impact parameter: b_crit = {b_crit:.4f}  (= 3√3·M ≈ {3*np.sqrt(3)*M_VAL:.4f} ✓)")

# Scan impact parameters: sub-critical (absorbed), near-critical, super-critical
b_values = np.array([b_crit*0.6, b_crit*0.85, b_crit*0.97,
                     b_crit*1.0, b_crit*1.03, b_crit*1.15, b_crit*1.4])
b_labels = [f'b={b/b_crit:.2f}b_c' for b in b_values]

# Spatial grid: r/r_s from just outside horizon to far field
r_grid = np.linspace(1.05, 6.0, 60)

# WKB state: incoming wave from large r  (plane wave in r*)
# Phase: S(y) = +ω·y  (ingoing, toward smaller r)
phase_in  = -OMEGA * y_sym / LAM   # WKB phase (divide by λ for normalisation)
amp_gauss = sp.exp(-((y_sym - 4.0)**2) / (2 * 1.2**2))   # Gaussian centred at r=4

part1_results = {}

for b_val, b_lab in zip(b_values, b_labels):
    p_sym_schw = schwarzschild_symbol(b_val)
    op_schw    = PseudoDifferentialOperator(p_sym_schw, vars_x=[x_sym], mode='symbol')
    bridge     = PsiOpFIOBridge(op_schw, **bridge_kw)

    with warnings.catch_warnings(record=True):
        warnings.simplefilter('always')
        u_out = bridge.evaluate_grid(r_grid, phase_in, amp_gauss)

    # WKB reference (no Maslov correction):
    #   p(r,k₀)·u₀(r)  where  k₀ = -ω/λ (ingoing momentum)
    k0      = -OMEGA / LAM
    p_vals  = np.array([float(p_sym_schw.subs([(x_sym, rv), (xi_sym, k0)]))
                        for rv in r_grid])
    u0_vals = (np.exp(-((r_grid - 4.0)**2)/(2*1.2**2))
               * np.exp(1j * LAM * (-OMEGA/LAM) * r_grid))
    u_wkb   = p_vals * u0_vals

    scale = np.max(np.abs(u_wkb)) + 1e-30
    err   = float(np.max(np.abs(u_out - u_wkb)) / scale)

    # Detect whether bridge found a near-degenerate critical point
    # by checking if any |u_out| peaks anomalously near r_lr
    lr_mask = np.abs(r_grid - r_lr_val) < 0.3
    amp_near_lr = float(np.max(np.abs(u_out[lr_mask]))) if lr_mask.any() else 0.
    amp_far     = float(np.max(np.abs(u_out[~lr_mask]))) if (~lr_mask).any() else 1.

    part1_results[b_lab] = dict(
        b_val       = b_val,
        u_bridge    = u_out,
        u_wkb       = u_wkb,
        err         = err,
        enhancement = amp_near_lr / (amp_far + 1e-30),
    )
    print(f"  {b_lab:20s}  err={err:.3e}  "
          f"lr_enhancement={amp_near_lr/(amp_far+1e-30):.2f}x")

# ═══════════════════════════════════════════════════════════════
# PART II — GRAVITATIONAL LENSING & EINSTEIN RING
# ═══════════════════════════════════════════════════════════════
print("\n" + "═"*60)
print("PART II — Gravitational lensing & Einstein ring")
print("═"*60)
print(f"  θ_E = {THETA_E:.4f}  (Einstein radius)")
print(f"  D_L = {D_L},  D_S = {D_S},  D_LS = {D_LS}")

# Lensing operator symbol:
#   p(y, xi) = xi^2  — trivially the free propagator
# but the WKB STATE carries the lensing phase:
#   S_source(y) = K_WAVE·y - θ_E²·ln|y|
# so that the stationary conditions reproduce the lens equation.
#
# IMPORTANT: ln|y| has a branch point at y=0.  We regularise:
#   S_lens(y) = -θ_E²·ln(sqrt(y²+ε²))  with ε=0.05
# This removes the singularity while preserving the physics for |y| >> ε.

EPSILON = 0.05   # regularisation of the point lens
THETA_E_sym = sp.Float(THETA_E)

# Lensing phase in source plane
#   S(y) = K_WAVE·y  -  θ_E²·(1/2)·ln(y²+ε²)
S_lens = (K_WAVE * y_sym
          - THETA_E_sym**2 * sp.Rational(1,2)
            * sp.log(y_sym**2 + EPSILON**2))

# Free propagation symbol  p = xi²
p_free = xi_sym**2
op_free = PseudoDifferentialOperator(p_free, vars_x=[x_sym], mode='symbol')

# Gaussian source envelope centred at y=0
amp_source = sp.exp(-y_sym**2 / (2 * 1.0**2))

# Observer grid: range of impact parameters b_obs
b_obs_grid = np.linspace(-3.0*THETA_E, 3.0*THETA_E, 80)

bridge_lens = PsiOpFIOBridge(op_free, **bridge_kw)
with warnings.catch_warnings(record=True):
    warnings.simplefilter('always')
    u_lensed = bridge_lens.evaluate_grid(b_obs_grid, S_lens, amp_source)

# Analytical magnification for each image:
#   μ±(b_obs) = (y± / b_obs) · (dy±/db_obs)^{-1}
# where  y± = (b_obs ± sqrt(b_obs²+4θ_E²)) / 2
def magnification(b_obs: np.ndarray) -> tuple:
    disc   = np.sqrt(b_obs**2 + 4*THETA_E**2)
    y_plus  = 0.5*(b_obs + disc)
    y_minus = 0.5*(b_obs - disc)
    # |μ| = (y/b_obs) / (1 - (θ_E/y)²)  (standard formula)
    mu_plus  = np.abs((y_plus  / (b_obs + 1e-15))
                      / (1 - (THETA_E/y_plus)**2))
    mu_minus = np.abs((y_minus / (b_obs + 1e-15))
                      / (1 - (THETA_E/y_minus)**2))
    return mu_plus, mu_minus, y_plus, y_minus

mu_p, mu_m, y_p, y_m = magnification(b_obs_grid)
mu_total_analytic     = mu_p + mu_m   # total magnification

# WKB reference (geometric optics limit, no interference):
# |u_GO(b_obs)|² ~ μ_total(b_obs)  (magnification = intensity ratio)
u_go_amp = np.sqrt(np.clip(mu_total_analytic, 0, 50))

# Deflection angle comparison at several impact parameters:
print("\n  Deflection angle check: α_GR = 4M/b vs bridge saddle position")
print(f"  {'b_obs':>8}  {'y+(analytic)':>14}  {'y-(analytic)':>14}  {'α_GR':>10}")
for b_val in [0.5, 1.0, 1.5, 2.0, 3.0]:
    y_pl = 0.5*(b_val + np.sqrt(b_val**2 + 4*THETA_E**2))
    y_mn = 0.5*(b_val - np.sqrt(b_val**2 + 4*THETA_E**2))
    alpha_val = 4*M_VAL / b_val
    print(f"  {b_val:>8.2f}  {y_pl:>14.4f}  {y_mn:>14.4f}  {alpha_val:>10.4f}")

# Einstein ring: at b_obs=0 the two images merge into a ring.
# The bridge should show a peak (Airy enhancement) at b_obs=0.
b_center_idx = np.argmin(np.abs(b_obs_grid))
u_ring = np.abs(u_lensed[b_center_idx])
u_away = np.mean(np.abs(u_lensed[np.abs(b_obs_grid) > 2*THETA_E]))
print(f"\n  Einstein ring enhancement: "
      f"|u(0)| / |u(far)| = {u_ring/(u_away+1e-30):.2f}x")

# Interference fringe spacing (two-image interference):
# Δφ = λ·(S(y+) - S(y-))  at a given b_obs.
# Fringe period in b_obs:  Δb ~ 2π/λ · (∂b_obs/∂Δφ)
print("\n  Interference fringe analysis:")
for b_val in [1.0, 2.0]:
    y_pl = 0.5*(b_val + np.sqrt(b_val**2 + 4*THETA_E**2))
    y_mn = 0.5*(b_val - np.sqrt(b_val**2 + 4*THETA_E**2))
    S_p  = K_WAVE*y_pl - THETA_E**2*0.5*np.log(y_pl**2 + EPSILON**2)
    S_m  = K_WAVE*y_mn - THETA_E**2*0.5*np.log(y_mn**2 + EPSILON**2)
    delta_phi = LAM * abs(S_p - S_m)
    print(f"  b_obs={b_val:.1f}: Δφ_WKB = {delta_phi:.2f} rad  "
          f"(fringe period ~ {2*np.pi/delta_phi:.3f} in b_obs units)")

# ═══════════════════════════════════════════════════════════════
# VISUALIZATION
# ═══════════════════════════════════════════════════════════════
DARK_BG  = '#0d1117'
GRID_CLR = '#21262d'
TICK_CLR = '#8b949e'
LBL_CLR  = '#c9d1d9'

# Colour ramp for Part I impact parameters
CMAP_I   = plt.cm.plasma
N_B      = len(b_values)
B_COLORS = [CMAP_I(i/(N_B-1)) for i in range(N_B)]

fig = plt.figure(figsize=(20, 16), facecolor=DARK_BG)
fig.suptitle(
    r'General Relativity + FIO Bridge'
    '\n'
    r'Part I: Schwarzschild radial symbol, light ring & Maslov phase  '
    r'|  Part II: Gravitational lensing, Einstein ring & wave-optics interference'
    f'\nλ={LAM:.0f},  M={M_VAL},  ω={OMEGA},  '
    f'b_crit=3√3·M={b_crit:.3f},  '
    f'θ_E={THETA_E:.3f}',
    color=LBL_CLR, fontsize=10, fontfamily='monospace', y=0.995,
)

gs = gridspec.GridSpec(
    3, 3, figure=fig,
    top=0.945, bottom=0.06,
    hspace=0.45, wspace=0.32,
    left=0.07, right=0.97,
)

# ── Panel I-A : Schwarzschild potential V_eff(r) ─────────────────────────
ax_Va = fig.add_subplot(gs[0, 0])
ax_Va.set_facecolor(DARK_BG)
for sp_ in ax_Va.spines.values(): sp_.set_edgecolor(GRID_CLR)
ax_Va.tick_params(colors=TICK_CLR, labelsize=7)

r_plot = np.linspace(1.01, 6.0, 400)
for i, (b_val, b_lab, clr) in enumerate(zip(b_values, b_labels, B_COLORS)):
    V_plot = OMEGA**2 * b_val**2 * (1 - 1/r_plot) / r_plot**2
    ax_Va.plot(r_plot, V_plot, color=clr, lw=1.5, label=b_lab)

# ω² horizontal line
ax_Va.axhline(OMEGA**2, color='white', lw=1.0, ls='--', alpha=0.7,
              label=f'ω²={OMEGA**2}')
# Light ring
ax_Va.axvline(r_lr_val, color='#e3b341', lw=1.2, ls=':',
              label=f'light ring r={r_lr_val:.2f}r_s')
ax_Va.axvline(1.0, color='#f85149', lw=1.0, ls='-', alpha=0.5,
              label='horizon r=r_s')

ax_Va.set_xlabel('r / r_s', color=LBL_CLR, fontsize=8)
ax_Va.set_ylabel('V_eff(r)', color=LBL_CLR, fontsize=8)
ax_Va.set_title('Schwarzschild effective potential', color=LBL_CLR,
                fontsize=9, fontfamily='monospace')
ax_Va.set_ylim(-1, OMEGA**2 * 2.5)
ax_Va.legend(fontsize=5, facecolor='#161b22', labelcolor=LBL_CLR,
             edgecolor=GRID_CLR, loc='upper right', ncol=1)
ax_Va.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)

# ── Panel I-B : Bridge output |u(r)| for all impact params ───────────────
ax_Vb = fig.add_subplot(gs[0, 1])
ax_Vb.set_facecolor(DARK_BG)
for sp_ in ax_Vb.spines.values(): sp_.set_edgecolor(GRID_CLR)
ax_Vb.tick_params(colors=TICK_CLR, labelsize=7)

for b_lab, clr in zip(b_labels, B_COLORS):
    r_ = part1_results[b_lab]
    ax_Vb.plot(r_grid, np.abs(r_['u_bridge']), color=clr, lw=1.5,
               label=b_lab)

ax_Vb.axvline(r_lr_val, color='#e3b341', lw=1.2, ls=':',
              label=f'light ring r={r_lr_val:.2f}r_s')
ax_Vb.axvline(1.0, color='#f85149', lw=1.0, ls='-', alpha=0.5)

ax_Vb.set_xlabel('r / r_s', color=LBL_CLR, fontsize=8)
ax_Vb.set_ylabel('|u(r)|  (bridge)', color=LBL_CLR, fontsize=8)
ax_Vb.set_title('Bridge output |u(r)| vs impact param', color=LBL_CLR,
                fontsize=9, fontfamily='monospace')
ax_Vb.legend(fontsize=5, facecolor='#161b22', labelcolor=LBL_CLR,
             edgecolor=GRID_CLR, loc='upper right', ncol=1)
ax_Vb.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)

# ── Panel I-C : Light-ring enhancement vs b ───────────────────────────────
ax_Vc = fig.add_subplot(gs[0, 2])
ax_Vc.set_facecolor(DARK_BG)
for sp_ in ax_Vc.spines.values(): sp_.set_edgecolor(GRID_CLR)
ax_Vc.tick_params(colors=TICK_CLR, labelsize=7)

b_arr  = np.array([part1_results[l]['b_val'] for l in b_labels])
enh    = np.array([part1_results[l]['enhancement'] for l in b_labels])

ax_Vc.plot(b_arr/b_crit, enh, 'o-', color='#e3b341', lw=1.8, ms=6)
ax_Vc.axvline(1.0, color='#e3b341', lw=1.2, ls=':',
              label='b = b_crit (light ring)')
ax_Vc.set_xlabel('b / b_crit', color=LBL_CLR, fontsize=8)
ax_Vc.set_ylabel('Enhancement near light ring', color=LBL_CLR, fontsize=8)
ax_Vc.set_title('Light-ring amplitude enhancement\n(Maslov/Airy caustic)',
                color=LBL_CLR, fontsize=9, fontfamily='monospace')
ax_Vc.legend(fontsize=7, facecolor='#161b22', labelcolor=LBL_CLR,
             edgecolor=GRID_CLR, loc='upper right')
ax_Vc.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)

# ── Panel II-A : Lensed field |u(b_obs)| — wave optics ───────────────────
ax_IIa = fig.add_subplot(gs[1, 0])
ax_IIa.set_facecolor(DARK_BG)
for sp_ in ax_IIa.spines.values(): sp_.set_edgecolor(GRID_CLR)
ax_IIa.tick_params(colors=TICK_CLR, labelsize=7)

ax_IIa.plot(b_obs_grid/THETA_E, np.abs(u_lensed),
            color='#3fb950', lw=1.8, label='wave optics (bridge)', zorder=3)
ax_IIa.plot(b_obs_grid/THETA_E, u_go_amp,
            color='#58a6ff', lw=1.4, ls='--',
            label='geometric optics √μ_tot', zorder=2)
ax_IIa.axvline(0, color='#e3b341', lw=1.0, ls=':',
               label='Einstein ring (b_obs=0)', alpha=0.8)
ax_IIa.axvline( 1, color='#388bfd', lw=0.8, ls=':', alpha=0.5)
ax_IIa.axvline(-1, color='#388bfd', lw=0.8, ls=':', alpha=0.5,
               label='b_obs = ±θ_E')

ax_IIa.set_xlabel('b_obs / θ_E', color=LBL_CLR, fontsize=8)
ax_IIa.set_ylabel('|u(b_obs)|', color=LBL_CLR, fontsize=8)
ax_IIa.set_title('Gravitational lensing: wave vs geometric optics',
                 color=LBL_CLR, fontsize=9, fontfamily='monospace')
ax_IIa.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR,
              edgecolor=GRID_CLR, loc='upper right')
ax_IIa.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)

# ── Panel II-B : Two-image positions vs b_obs ─────────────────────────────
ax_IIb = fig.add_subplot(gs[1, 1])
ax_IIb.set_facecolor(DARK_BG)
for sp_ in ax_IIb.spines.values(): sp_.set_edgecolor(GRID_CLR)
ax_IIb.tick_params(colors=TICK_CLR, labelsize=7)

b_dense = np.linspace(-3*THETA_E, 3*THETA_E, 300)
_, _, yp_dense, ym_dense = magnification(b_dense)

ax_IIb.plot(b_dense/THETA_E, yp_dense/THETA_E, color='#f78166', lw=2.0,
            label='image y+ / θ_E')
ax_IIb.plot(b_dense/THETA_E, ym_dense/THETA_E, color='#d2a8ff', lw=2.0,
            label='image y− / θ_E')
ax_IIb.plot(b_dense/THETA_E, b_dense/THETA_E, color='white', lw=0.8,
            ls='--', alpha=0.4, label='y = b (no lensing)')
ax_IIb.axhline( 1, color='#e3b341', lw=0.8, ls=':', alpha=0.6)
ax_IIb.axhline(-1, color='#e3b341', lw=0.8, ls=':', alpha=0.6,
               label='y = ±θ_E')
ax_IIb.axvline(0, color='#e3b341', lw=0.8, ls=':', alpha=0.5)

ax_IIb.set_xlabel('b_obs / θ_E', color=LBL_CLR, fontsize=8)
ax_IIb.set_ylabel('image position y / θ_E', color=LBL_CLR, fontsize=8)
ax_IIb.set_title('Lens equation: image positions\n(two FIO critical points)',
                 color=LBL_CLR, fontsize=9, fontfamily='monospace')
ax_IIb.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR,
              edgecolor=GRID_CLR, loc='upper left')
ax_IIb.set_ylim(-4, 4)
ax_IIb.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)

# ── Panel II-C : Magnification curves μ±(b_obs) ───────────────────────────
ax_IIc = fig.add_subplot(gs[1, 2])
ax_IIc.set_facecolor(DARK_BG)
for sp_ in ax_IIc.spines.values(): sp_.set_edgecolor(GRID_CLR)
ax_IIc.tick_params(colors=TICK_CLR, labelsize=7)

b_mag   = b_dense[np.abs(b_dense) > 0.15*THETA_E]  # avoid b=0 divergence
_, _, yp_m, ym_m = magnification(b_mag)
mu_p_m  = np.abs((yp_m / (b_mag+1e-15)) / (1 - (THETA_E/yp_m)**2))
mu_m_m  = np.abs((ym_m / (b_mag+1e-15)) / (1 - (THETA_E/ym_m)**2))
mu_tot_m = mu_p_m + mu_m_m

ax_IIc.semilogy(b_mag/THETA_E, mu_p_m,  color='#f78166', lw=1.8,
                label='μ+ (bright image)')
ax_IIc.semilogy(b_mag/THETA_E, mu_m_m,  color='#d2a8ff', lw=1.8,
                label='μ− (faint image)')
ax_IIc.semilogy(b_mag/THETA_E, mu_tot_m, color='#3fb950', lw=1.4,
                ls='--', label='μ_tot = μ+ + μ−')
ax_IIc.axvline(0, color='#e3b341', lw=1.0, ls=':', alpha=0.8,
               label='Einstein ring (μ→∞)')
ax_IIc.axhline(1.34, color='white', lw=0.8, ls='--', alpha=0.4,
               label='μ=1.34 at b=θ_E')

ax_IIc.set_xlabel('b_obs / θ_E', color=LBL_CLR, fontsize=8)
ax_IIc.set_ylabel('magnification μ', color=LBL_CLR, fontsize=8)
ax_IIc.set_title('Gravitational lens magnification',
                 color=LBL_CLR, fontsize=9, fontfamily='monospace')
ax_IIc.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR,
              edgecolor=GRID_CLR, loc='upper right')
ax_IIc.set_ylim(0.1, 50)
ax_IIc.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)

# ── Panel III : Phase of lensed field — interference fringes ─────────────
ax_IIIa = fig.add_subplot(gs[2, 0])
ax_IIIa.set_facecolor(DARK_BG)
for sp_ in ax_IIIa.spines.values(): sp_.set_edgecolor(GRID_CLR)
ax_IIIa.tick_params(colors=TICK_CLR, labelsize=7)

ax_IIIa.plot(b_obs_grid/THETA_E, np.angle(u_lensed),
             color='#58a6ff', lw=1.5, label='arg u(b_obs)', zorder=3)
ax_IIIa.axvline(0, color='#e3b341', lw=1.0, ls=':', alpha=0.8)

ax_IIIa.set_xlabel('b_obs / θ_E', color=LBL_CLR, fontsize=8)
ax_IIIa.set_ylabel('phase  arg u(b_obs)', color=LBL_CLR, fontsize=8)
ax_IIIa.set_title('Phase of lensed field\n(interference between two images)',
                  color=LBL_CLR, fontsize=9, fontfamily='monospace')
ax_IIIa.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR,
               edgecolor=GRID_CLR, loc='lower right')
ax_IIIa.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)

# ── Panel III-B : |u|² (intensity) with fringe visibility ────────────────
ax_IIIb = fig.add_subplot(gs[2, 1])
ax_IIIb.set_facecolor(DARK_BG)
for sp_ in ax_IIIb.spines.values(): sp_.set_edgecolor(GRID_CLR)
ax_IIIb.tick_params(colors=TICK_CLR, labelsize=7)

intensity_bridge = np.abs(u_lensed)**2
intensity_go     = u_go_amp**2

ax_IIIb.plot(b_obs_grid/THETA_E, intensity_bridge,
             color='#3fb950', lw=1.8, label='|u|² wave optics', zorder=3)
ax_IIIb.plot(b_obs_grid/THETA_E, intensity_go,
             color='#58a6ff', lw=1.4, ls='--',
             label='μ_tot geometric optics', zorder=2)
ax_IIIb.fill_between(b_obs_grid/THETA_E, intensity_bridge,
                     alpha=0.1, color='#3fb950')
ax_IIIb.axvline(0, color='#e3b341', lw=1.0, ls=':', alpha=0.8,
                label='Einstein ring')

ax_IIIb.set_xlabel('b_obs / θ_E', color=LBL_CLR, fontsize=8)
ax_IIIb.set_ylabel('intensity |u|²', color=LBL_CLR, fontsize=8)
ax_IIIb.set_title('Wave vs geometric optics intensity\n(fringes from two-image interference)',
                  color=LBL_CLR, fontsize=9, fontfamily='monospace')
ax_IIIb.legend(fontsize=6, facecolor='#161b22', labelcolor=LBL_CLR,
               edgecolor=GRID_CLR, loc='upper right')
ax_IIIb.set_ylim(bottom=0)
ax_IIIb.grid(True, color=GRID_CLR, lw=0.5, alpha=0.7)

# ── Panel III-C : deflection angle check ─────────────────────────────────
ax_IIIc = fig.add_subplot(gs[2, 2])
ax_IIIc.set_facecolor(DARK_BG)
for sp_ in ax_IIIc.spines.values(): sp_.set_edgecolor(GRID_CLR)
ax_IIIc.tick_params(colors=TICK_CLR, labelsize=7)

b_check = np.linspace(0.3*THETA_E, 4*THETA_E, 200)
alpha_gr = 4*M_VAL / b_check                # Einstein formula
# Deflection angle from lens equation: α(b) = (b - b_obs)/D_LS * D_S/D_LS
# At the FIO saddle y_c(b_obs): deflection = y_c - b_obs
_, _, yp_c, _ = magnification(b_check)
alpha_bridge = (yp_c - b_check) / b_check   # normalised deflection

ax_IIIc.loglog(b_check/THETA_E, alpha_gr,    color='#58a6ff', lw=2.0,
               label='α = 4M/b  (Einstein)', zorder=3)
ax_IIIc.loglog(b_check/THETA_E, np.abs(alpha_bridge), color='#f78166',
               lw=1.6, ls='--', label='α from lens equation', zorder=2)

ax_IIIc.set_xlabel('b / θ_E', color=LBL_CLR, fontsize=8)
ax_IIIc.set_ylabel('deflection angle α  (log)', color=LBL_CLR, fontsize=8)
ax_IIIc.set_title('Deflection angle: Einstein formula\nvs lens equation saddle',
                  color=LBL_CLR, fontsize=9, fontfamily='monospace')
ax_IIIc.legend(fontsize=7, facecolor='#161b22', labelcolor=LBL_CLR,
               edgecolor=GRID_CLR, loc='lower left')
ax_IIIc.grid(True, color=GRID_CLR, lw=0.5, alpha=0.5, which='both')

plt.show()

# ═══════════════════════════════════════════════════════════════
# CONSOLE SUMMARY
# ═══════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("SUMMARY — GR + FIO Bridge")
print("═"*65)
print(f"\n  PART I — Schwarzschild")
print(f"  M={M_VAL}, r_s={RS}, ω={OMEGA}, b_crit=3√3·M={b_crit:.4f}")
print(f"  Light ring at r/r_s = {r_lr_val} = 3M ✓")
print()
print(f"  {'Impact param':>22}  {'b/b_crit':>8}  {'LR enhance':>12}  {'err':>10}")
print(f"  {'-'*22}  {'-'*8}  {'-'*12}  {'-'*10}")
for b_lab, clr in zip(b_labels, B_COLORS):
    r_ = part1_results[b_lab]
    print(f"  {b_lab:>22}  {r_['b_val']/b_crit:>8.3f}  "
          f"{r_['enhancement']:>12.3f}  {r_['err']:>10.3e}")

print(f"\n  PART II — Gravitational lensing")
print(f"  θ_E={THETA_E:.4f}, D_L={D_L}, D_S={D_S}, D_LS={D_LS}")
print(f"  Einstein ring enhancement: {u_ring/(u_away+1e-30):.2f}x")
print(f"\n  Bridge critical points = gravitational lens images:")
print(f"  {'b_obs':>8}  {'y+ analytic':>14}  {'y- analytic':>14}")
for b_val in [0.5, 1.0, 2.0]:
    y_pl = 0.5*(b_val + np.sqrt(b_val**2 + 4*THETA_E**2))
    y_mn = 0.5*(b_val - np.sqrt(b_val**2 + 4*THETA_E**2))
    print(f"  {b_val:>8.2f}  {y_pl:>14.4f}  {y_mn:>14.4f}")

## Cross-Validation

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from fio_bridge import CrossValidator, WKBState
from psiop import PseudoDifferentialOperator

# 1. Define BOTH x and y symbols
x_sym = sp.Symbol('x', real=True)
y_sym = sp.Symbol('y', real=True)  # <--- CRITICAL FIX: Use 'y' for the WKB state

# 2. Define the Operator using x and xi
xi_sym = sp.Symbol('xi', real=True)
op_sym = xi_sym**2 + 1
P = PseudoDifferentialOperator(op_sym, vars_x=[x_sym])
lam = 20.0
L = 5.0                         # domain length
oversampling = 3                  # safety margin (2 is a common choice)
Nx = int(oversampling * lam * L / np.pi)

# If you prefer to set λ based on a fixed grid, use:
# lam_max = np.pi * Nx / (oversampling * L)
# lam = min(desired_lam, lam_max)   # ensure λ does not exceed resolution
# 3. Define the WKB State using y_sym
# This perfectly aligns with PsiOpFIOBridge's internal integration logic
u0_wkb = WKBState(
    amp_sym   = sp.exp(-y_sym**2),    # Now using y_sym
    phase_sym = sp.cos(y_sym),        # Now using y_sym
    var_x     = y_sym,                # Bind the state to y_sym
    lam       = lam
)

# 4. Setup the grid
x_grid = np.linspace(-L/2, L/2, Nx)

# 5. Initialize and Run Validator
validator = CrossValidator(
    op=P, 
    wkb_state=u0_wkb, 
    x_grid=x_grid, 
    lam=lam
)
report = validator.run()

# 6. Extract metrics using proper dataclass attributes
print(f"Max Relative Error: {report.max_rel_error:.2e}")
print(f"Validation Passed: {report.wkb_valid}")  # <--- Use .wkb_valid

# 7. Plotting
validator.plot_report(report, title=fr"Cross-Validation for $\lambda={lam}$")


## Semi-classical Corrector & Cross-Validation V1

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from psiop import PseudoDifferentialOperator
from fio_bridge import (
    CrossValidator, WKBState, 
    SpectralSplitter, SemiclassicalCorrector
)

# ------------------------------------------------------------
# 1. Define the operator and WKB state (same as your example)
# ------------------------------------------------------------
x_sym = sp.Symbol('x', real=True)
y_sym = sp.Symbol('y', real=True)      # WKB state uses 'y'
xi_sym = sp.Symbol('xi', real=True)

op_sym = xi_sym**2 + 1
P = PseudoDifferentialOperator(op_sym, vars_x=[x_sym])

lam = 20.0
L = 10.0
oversampling = 2
Nx = int(oversampling * lam * L / np.pi)
x_grid = np.linspace(-L/2, L/2, Nx)

u0_wkb = WKBState(
    amp_sym   = sp.exp(-(y_sym**2 + 1)),
    phase_sym = sp.cos(y_sym),
    var_x     = y_sym,
    lam       = lam
)

# ------------------------------------------------------------
# 2. Run CrossValidator to get both solver and bridge solutions
# ------------------------------------------------------------
validator = CrossValidator(
    op=P, wkb_state=u0_wkb, x_grid=x_grid, lam=lam
)
report = validator.run()   # report.u_solver and report.u_bridge

# ------------------------------------------------------------
# 3. Set up a SpectralSplitter with a sensible k_cut
#    Use the dominant wavenumber of the WKB state
# ------------------------------------------------------------
k_cut = u0_wkb.dominant_wavenumber(x_grid)  # median |k|
splitter = SpectralSplitter(x_grid, k_cut=k_cut)

# ------------------------------------------------------------
# 4. Create the corrector and apply it to the solver solution
# ------------------------------------------------------------
corrector = SemiclassicalCorrector(
    P, splitter, lam=lam, n_guesses=50,
    xi_range=(-10, 10), y_range=(-6, 6)
)
u_corrected = corrector.correct(report.u_solver, u0_wkb)

# ------------------------------------------------------------
# 5. Evaluate errors
# ------------------------------------------------------------
err_solver = np.abs(report.u_solver - report.u_bridge)
err_corrected = np.abs(u_corrected - report.u_bridge)

max_err_solver = np.max(err_solver)
max_err_corrected = np.max(err_corrected)
improvement = (max_err_solver - max_err_corrected) / max_err_solver * 100

# ------------------------------------------------------------
# 6. Plot results
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

ax = axes[0, 0]
ax.plot(x_grid, np.real(report.u_bridge), 'k-', lw=1.5, label='Bridge')
ax.plot(x_grid, np.real(report.u_solver), 'r--', lw=1.5, label='Solver')
ax.plot(x_grid, np.real(u_corrected), 'b:', lw=1.5, label='Corrected')
ax.set_title('Real part comparison')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.semilogy(x_grid, err_solver, 'r-', label='Solver error')
ax.semilogy(x_grid, err_corrected, 'b-', label='Corrected error')
ax.set_title(f'Error |soln − bridge|')
ax.set_xlabel('x')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
# Show low/high split
u_low, u_high = splitter.split(report.u_solver)
ax.plot(x_grid, np.abs(u_low), label='|low|', alpha=0.7)
ax.plot(x_grid, np.abs(u_high), label='|high|', alpha=0.7)
ax.set_title('Solver solution decomposition')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.text(0.1, 0.5,
        f"Max error (solver)   = {max_err_solver:.2e}\n"
        f"Max error (corrected)= {max_err_corrected:.2e}\n"
        f"Improvement = {improvement:.1f}%",
        transform=ax.transAxes, fontsize=10)
ax.set_title('Error summary')
ax.axis('off')

plt.tight_layout()
plt.show()

## Semi-classical Corrector & Cross-Validation V2

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from psiop import PseudoDifferentialOperator
from fio_bridge import (
    CrossValidator, WKBState, SpectralSplitter, SemiclassicalCorrector
)

# --- same setup as before ---
x_sym, y_sym, xi_sym = sp.symbols('x y xi', real=True)
op_sym = xi_sym**2 + 1
P = PseudoDifferentialOperator(op_sym, vars_x=[x_sym])

lam = 20.0
L = 10.0
oversampling = 2
Nx = int(oversampling * lam * L / np.pi)
x_grid = np.linspace(-L/2, L/2, Nx)

u0_wkb = WKBState(
    amp_sym=sp.exp(-(y_sym**2 + 1)),
    phase_sym=sp.cos(y_sym),
    var_x=y_sym,
    lam=lam
)

# --- run cross-validator to get baseline solutions ---
validator = CrossValidator(P, u0_wkb, x_grid, lam)
report = validator.run()

# --- adaptive splitter based on bridge's energy ---
splitter = SpectralSplitter(x_grid)
target_high_fraction=0.9
k_cut = splitter.suggest_k_cut(report.u_bridge, target_high_fraction=target_high_fraction)
splitter.k_cut = k_cut
splitter._mask_low = np.abs(splitter.k_grid) <= k_cut
splitter._mask_high = ~splitter._mask_low

# --- corrector with iterative refinement ---
corrector = SemiclassicalCorrector(P, splitter, lam=lam)
u_corrected = report.u_solver.copy()
for _ in range(2):
    u_corrected = corrector.correct(u_corrected, u0_wkb)

# --- plot results ---
err_solver = np.abs(report.u_solver - report.u_bridge)
err_corrected = np.abs(u_corrected - report.u_bridge)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].semilogy(x_grid, err_solver, 'r-', label='solver error')
ax[0].semilogy(x_grid, err_corrected, 'b-', label='corrected error')
ax[0].set_title('Error vs bridge')
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(x_grid, np.real(report.u_bridge), 'k-', lw=1.5, label='bridge')
ax[1].plot(x_grid, np.real(report.u_solver), 'r--', lw=1.5, label='solver')
ax[1].plot(x_grid, np.real(u_corrected), 'b:', lw=1.5, label='corrected')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max solver error: {np.max(err_solver):.2e}")
print(f"Max corrected error: {np.max(err_corrected):.2e}")
print(f"Improvement factor: {np.max(err_solver)/np.max(err_corrected):.2f}")